In [1]:
"""
═══════════════════════════════
Central configuration, all typed data structures, theme enums, and
terminal logger for the Theme-Aligned RAG system.

Theme-Aligned RAG Novelty:
  All prior systems retrieve by semantic similarity: "find chunks most
  similar to this query vector." Theme-Aligned RAG retrieves by thematic
  coherence: first discover the latent thematic structure of the corpus,
  then align queries to themes before retrieving chunks.

  Every document chunk is assigned a soft theme distribution
  P(theme_k | chunk) — not a hard label. Queries are projected into the
  same theme space. Retrieval combines:
    • semantic relevance   — cosine(query, chunk)
    • thematic alignment   — KL-divergence(query_theme_dist, chunk_theme_dist)
    • thematic coherence   — within-theme consistency of retrieved set

  For multi-theme queries, Theme-Aligned RAG runs per-theme sub-retrievals
  and assembles a thematically structured answer with explicit narrative
  threading: Introduction → Theme A context → Theme B context → Synthesis.

References:
  • BERTopic (Grootendorst, 2022)
    https://arxiv.org/abs/2203.05794
  • LDA (Blei et al., 2003)
    https://dl.acm.org/doi/10.5555/944919.944937
  • TopicGPT (Pham et al., 2023)
    https://arxiv.org/abs/2311.01449
  • CSTM: Coherence-Sensitive Topic Models (Li & McCallum, 2006)
    https://dl.acm.org/doi/10.5555/1220175.1220243
  • Narrative Coherence in NLG (Barzilay & Lapata, 2008)
    https://doi.org/10.1162/coli.2008.34.1.1
  • Theme-Guided RAG survey (Shen et al., 2024)
    https://arxiv.org/abs/2401.13192
  • CTRLsum (He et al., 2022) — controlled summarization
    https://arxiv.org/abs/2012.04281
  • Topic-Aware Retrieval (Zhao et al., 2021)
    https://arxiv.org/abs/2104.07597
"""


'\n═══════════════════════════════\nCentral configuration, all typed data structures, theme enums, and\nterminal logger for the Theme-Aligned RAG system.\n\nTheme-Aligned RAG Novelty:\n  All prior systems retrieve by semantic similarity: "find chunks most\n  similar to this query vector." Theme-Aligned RAG retrieves by thematic\n  coherence: first discover the latent thematic structure of the corpus,\n  then align queries to themes before retrieving chunks.\n\n  Every document chunk is assigned a soft theme distribution\n  P(theme_k | chunk) — not a hard label. Queries are projected into the\n  same theme space. Retrieval combines:\n    • semantic relevance   — cosine(query, chunk)\n    • thematic alignment   — KL-divergence(query_theme_dist, chunk_theme_dist)\n    • thematic coherence   — within-theme consistency of retrieved set\n\n  For multi-theme queries, Theme-Aligned RAG runs per-theme sub-retrievals\n  and assembles a thematically structured answer with explicit narrative\n  th

In [2]:

from __future__ import annotations

import os
import math
import time
from enum import Enum
from typing import Any, Dict, List, Optional, Set, Tuple
from dataclasses import dataclass, field


In [40]:

# ══════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════

@dataclass
class Config:
    # ── Ollama (local LLM) ─────────────────────────────────────────────
    OLLAMA_BASE_URL: str = field(default_factory=lambda: os.getenv(
        "OLLAMA_BASE_URL", "http://localhost:11434"))
    OLLAMA_MODEL: str    = field(default_factory=lambda: os.getenv(
        "OLLAMA_MODEL", "qwen2.5-coder:7b"))

    # ── HuggingFace Embeddings (local) ───────────────────────────────
    EMBEDDING_MODEL: str  = "sentence-transformers/all-MiniLM-L6-v2"
    EMBEDDING_DEVICE: str = field(default_factory=lambda: os.getenv("EMBEDDING_DEVICE", "cpu"))

    # ── Storage ───────────────────────────────────────────────────────
    BASE_DIR: str          = "./theme_rag_store"
    CHROMA_DIR: str        = "./theme_rag_store/chroma"
    THEMES_PATH: str       = "./theme_rag_store/themes.json"
    CHUNK_DIST_PATH: str   = "./theme_rag_store/chunk_theme_dists.json"
    COLLECTION_NAME: str   = "theme_rag_chunks"

    # ── Chunking ──────────────────────────────────────────────────────
    CHUNK_SIZE: int        = 600
    CHUNK_OVERLAP: int     = 120

    # ── Theme Discovery ───────────────────────────────────────────────
    NUM_THEMES: int        = 8       # K = number of latent themes to discover
    TOP_WORDS_PER_THEME: int = 12    # descriptor vocabulary per theme
    MIN_THEME_COHERENCE: float = 0.3 # minimum coherence score to keep a theme

    # ── Theme Alignment ───────────────────────────────────────────────
    # How strongly theme alignment re-ranks vs raw semantic score
    THEME_WEIGHT: float    = 0.40    # 0 = pure semantic, 1 = pure thematic
    SEMANTIC_WEIGHT: float = 0.60    # complement of THEME_WEIGHT
    # Query threshold for triggering multi-theme retrieval
    MULTI_THEME_THRESHOLD: float = 0.25  # if 2nd theme prob ≥ this, use multi-theme
    # Thematic drift: if retrieved set coherence < this, trigger re-retrieval
    DRIFT_THRESHOLD: float = 0.35

    # ── Retrieval ─────────────────────────────────────────────────────
    TOP_K_SEMANTIC: int    = 12   # initial semantic candidates
    TOP_K_FINAL: int       = 6    # after theme re-ranking
    TOP_K_PER_THEME: int   = 4    # per theme in multi-theme retrieval
    FETCH_K: int           = 20
    MMR_LAMBDA: float      = 0.65

    # ── Narrative Threading ───────────────────────────────────────────
    MAX_NARRATIVE_THEMES: int = 3   # max themes woven into narrative answer
    NARRATIVE_TRANSITIONS: bool = True  # insert transition sentences between themes

    # ── LLM ──────────────────────────────────────────────────────────
    LLM_TEMPERATURE: float = 0.0
    LLM_MAX_TOKENS: int    = 1400

    def is_configured(self) -> bool:
        """Check Ollama is reachable."""
        import urllib.request
        try:
            urllib.request.urlopen(self.OLLAMA_BASE_URL, timeout=3)
            return True
        except Exception:
            return False
# ══════════════════════════════════════════════════════════════════════
# ENUMS
# ══════════════════════════════════════════════════════════════════════

class ThemeStatus(str, Enum):
    ALIGNED    = "aligned"     # query cleanly maps to one theme
    MULTI      = "multi"       # query spans multiple themes
    AMBIGUOUS  = "ambiguous"   # no clear theme dominance
    NOVEL      = "novel"       # query falls outside all discovered themes


class RetrievalMode(str, Enum):
    SEMANTIC_ONLY  = "semantic_only"   # baseline: no theme alignment
    THEME_ALIGNED  = "theme_aligned"   # single-theme alignment
    MULTI_THEME    = "multi_theme"     # cross-theme retrieval + synthesis
    NARRATIVE      = "narrative"       # narrative threading across themes
    ADAPTIVE       = "adaptive"        # auto-select mode by query analysis


class NarrativeArc(str, Enum):
    PROBLEM_SOLUTION = "problem_solution"    # problem → analysis → solution
    COMPARE_CONTRAST = "compare_contrast"    # entity A vs entity B
    CHRONOLOGICAL    = "chronological"       # timeline / evolution
    CAUSAL_CHAIN     = "causal_chain"        # cause → effect → implication
    GENERAL_SPECIFIC = "general_specific"    # overview → details → examples
    FREE_FORM        = "free_form"           # no predefined arc


# ══════════════════════════════════════════════════════════════════════
# CORE TYPED DATACLASSES
# ══════════════════════════════════════════════════════════════════════

@dataclass
class Theme:
    """
    A discovered latent theme in the corpus.
    Analogous to a topic in LDA, but named and described by an LLM.

    theme_id        : unique ID (e.g. "T0", "T3")
    label           : short LLM-generated name (e.g. "Attention Mechanisms")
    description     : 1–2 sentence LLM summary of the theme
    top_words       : most characteristic vocabulary for this theme
    centroid        : mean embedding of all chunks assigned to this theme
    coherence_score : average cosine similarity within theme (0–1)
    chunk_ids       : IDs of all chunks primarily assigned to this theme
    """
    theme_id:        str
    label:           str
    description:     str              = ""
    top_words:       List[str]        = field(default_factory=list)
    centroid:        Optional[List[float]] = None   # embedding vector
    coherence_score: float            = 0.0
    chunk_ids:       List[str]        = field(default_factory=list)
    doc_sources:     List[str]        = field(default_factory=list)
    metadata:        Dict[str, Any]   = field(default_factory=dict)

    def to_context_str(self) -> str:
        words = ", ".join(self.top_words[:8])
        return (f"[{self.theme_id}] {self.label}: {self.description} "
                f"(keywords: {words}; coherence={self.coherence_score:.2f})")

    def __repr__(self) -> str:
        return f"Theme({self.theme_id}: '{self.label}', coh={self.coherence_score:.2f})"


@dataclass
class ThemeDistribution:
    """
    Soft theme assignment for a chunk or query.
    P(theme_k | item) for all K themes. Sums to 1.0.
    Analogous to a document's topic distribution in LDA.
    """
    item_id:       str
    distribution:  Dict[str, float]    # theme_id → probability
    primary_theme: str                 # argmax theme_id
    secondary_theme: Optional[str]     = None  # 2nd highest if > MULTI_THEME_THRESHOLD
    entropy:       float               = 0.0   # distribution entropy (high = diffuse)

    def kl_divergence(self, other: "ThemeDistribution") -> float:
        """
        KL divergence D_KL(self || other) — measures thematic distance.
        Lower = more thematically aligned.
        Uses add-ε smoothing to avoid log(0).
        """
        eps = 1e-9
        div = 0.0
        all_keys = set(self.distribution) | set(other.distribution)
        for k in all_keys:
            p = self.distribution.get(k, eps)
            q = other.distribution.get(k, eps)
            div += p * math.log(p / (q + eps) + eps)
        return max(0.0, div)

    def theme_similarity(self, other: "ThemeDistribution") -> float:
        """
        Soft cosine similarity between two theme distributions.
        1 = perfectly aligned, 0 = completely different themes.
        """
        dot = sum(
            self.distribution.get(k, 0.0) * other.distribution.get(k, 0.0)
            for k in self.distribution
        )
        norm_self  = math.sqrt(sum(v**2 for v in self.distribution.values()))
        norm_other = math.sqrt(sum(v**2 for v in other.distribution.values()))
        denom = (norm_self * norm_other) + 1e-9
        return dot / denom

    def top_themes(self, n: int = 3) -> List[Tuple[str, float]]:
        """Return top-n (theme_id, probability) pairs."""
        return sorted(self.distribution.items(), key=lambda x: x[1], reverse=True)[:n]


@dataclass
class ThemeAlignedChunk:
    """
    A document chunk enriched with theme distribution metadata.
    Stored in ChromaDB alongside its semantic embedding.
    """
    chunk_id:         str
    content:          str
    doc_id:           str
    source:           str
    theme_dist:       ThemeDistribution
    semantic_score:   float = 0.0    # cosine similarity to query
    thematic_score:   float = 0.0    # theme alignment to query
    combined_score:   float = 0.0    # weighted combination
    page_num:         Optional[int] = None
    metadata:         Dict[str, Any] = field(default_factory=dict)

    def to_citation(self) -> str:
        loc = f"p.{self.page_num}" if self.page_num else "§chunk"
        return f"[{self.source} · {loc} · {self.theme_dist.primary_theme}]"


@dataclass
class QueryAnalysis:
    """
    Full analysis of a query before retrieval.
    Determines which themes are relevant and which retrieval mode to use.
    """
    query:             str
    theme_dist:        Optional[ThemeDistribution] = None
    primary_theme:     Optional[Theme]             = None
    secondary_themes:  List[Theme]                 = field(default_factory=list)
    status:            str                         = ThemeStatus.AMBIGUOUS.value
    retrieval_mode:    str                         = RetrievalMode.ADAPTIVE.value
    narrative_arc:     str                         = NarrativeArc.FREE_FORM.value
    hyde_passage:      str                         = ""    # hypothetical passage (optional)
    expanded_query:    str                         = ""    # theme-expanded query


@dataclass
class NarrativeThread:
    """
    A thematically structured narrative built from retrieved chunks.
    Defines the arc and inter-theme transitions for the generated answer.
    """
    arc:          NarrativeArc
    theme_order:  List[str]       # ordered theme_ids for the narrative
    sections:     Dict[str, List[ThemeAlignedChunk]]  # theme_id → chunks for this section
    transitions:  Dict[str, str]  # "T1→T2" → transition sentence
    intro:        str             = ""
    conclusion:   str             = ""


@dataclass
class ThemeAlignedResult:
    """Full result from a Theme-Aligned RAG query."""
    query:             str
    mode:              RetrievalMode
    query_analysis:    Optional[QueryAnalysis]          = None
    retrieved_chunks:  List[ThemeAlignedChunk]          = field(default_factory=list)
    narrative_thread:  Optional[NarrativeThread]        = None
    answer:            str                              = ""
    theme_report:      str                              = ""   # which themes contributed
    drift_detected:    bool                             = False
    drift_corrected:   bool                             = False
    retrieval_rounds:  int                              = 1
    themes_used:       List[str]                        = field(default_factory=list)
    elapsed_sec:       float                            = 0.0

    def unique_sources(self) -> List[str]:
        return sorted({c.source for c in self.retrieved_chunks})


# ══════════════════════════════════════════════════════════════════════
# TERMINAL LOGGER
# ══════════════════════════════════════════════════════════════════════

THEME_PALETTE = [
    "\033[96m",  # cyan
    "\033[92m",  # green
    "\033[93m",  # yellow
    "\033[95m",  # magenta
    "\033[94m",  # blue
    "\033[91m",  # red
    "\033[33m",  # orange
    "\033[97m",  # white
]


class C:
    RESET  = "\033[0m"; BOLD  = "\033[1m"; DIM   = "\033[2m"
    CYAN   = "\033[96m"; GREEN = "\033[92m"; YELLOW = "\033[93m"
    RED    = "\033[91m"; MAG   = "\033[95m"; BLUE   = "\033[94m"
    WHITE  = "\033[97m"; ORANGE= "\033[33m"
    THEME  = "\033[95m"    # magenta  — theme operations
    ALIGN  = "\033[92m"    # green    — alignment scoring
    DRIFT  = "\033[91m"    # red      — drift detection
    NARR   = "\033[94m"    # blue     — narrative threading
    QUERY  = "\033[96m"    # cyan     — query analysis
    DISC   = "\033[93m"    # yellow   — theme discovery
    MACRO  = THEME          # compatibility alias
    MESO   = ALIGN          # compatibility alias
    THREAD = NARR           # compatibility alias


W = 76


class Log:
    @staticmethod
    def banner(title: str, sub: str = ""):
        print(f"\n{C.BOLD}{C.WHITE}{'═'*W}")
        print(f"  {title}")
        if sub:
            print(f"  {C.DIM}{sub}{C.RESET}{C.BOLD}{C.WHITE}")
        print(f"{'═'*W}{C.RESET}")

    @staticmethod
    def section(title: str, colour: str = C.CYAN):
        print(f"\n{C.BOLD}{colour}{'━'*W}")
        print(f"  {title}")
        print(f"{'━'*W}{C.RESET}")

    @staticmethod
    def theme_row(theme: "Theme"):
        idx   = int(theme.theme_id[1:]) % len(THEME_PALETTE)
        col   = THEME_PALETTE[idx]
        words = " · ".join(theme.top_words[:6])
        print(f"  {C.BOLD}{col}[{theme.theme_id}]{C.RESET} "
              f"{C.BOLD}{theme.label:30s}{C.RESET} "
              f"{C.DIM}coh={theme.coherence_score:.2f}  chunks={len(theme.chunk_ids):3d}"
              f"  {words}{C.RESET}")

    @staticmethod
    def theme(theme: "Theme"):
        # Backward-compatible alias used elsewhere in the notebook
        Log.theme_row(theme)

    @staticmethod
    def step(msg: str):
        print(f"\n{C.BOLD}{C.BLUE}▶ {msg}{C.RESET}")

    @staticmethod
    def ok(msg: str):
        print(f"{C.GREEN}  ✓ {msg}{C.RESET}")

    @staticmethod
    def warn(msg: str):
        print(f"{C.YELLOW}  ⚠  {msg}{C.RESET}")

    @staticmethod
    def err(msg: str):
        print(f"{C.RED}  ✗ {msg}{C.RESET}")

    @staticmethod
    def info(msg: str):
        print(f"{C.DIM}  · {msg}{C.RESET}")

    @staticmethod
    def alignment(chunk_id: str, sem: float, them: float, combined: float,
                  theme_id: str):
        print(f"  {C.DIM}{chunk_id[:10]}{C.RESET}  "
              f"sem={C.CYAN}{sem:.3f}{C.RESET}  "
              f"theme={C.THEME}{them:.3f}{C.RESET}  "
              f"combined={C.ALIGN}{combined:.3f}{C.RESET}  "
              f"{C.DIM}[{theme_id}]{C.RESET}")

    @staticmethod
    def drift(score: float, threshold: float):
        col = C.RED if score < threshold else C.GREEN
        bar_len = 24
        filled  = int(bar_len * score)
        bar     = "█" * filled + "░" * (bar_len - filled)
        print(f"\n  {C.DRIFT}Coherence: {col}{bar}{C.RESET} "
              f"{score:.2f} (threshold={threshold:.2f}) "
              f"{'⚠ DRIFT DETECTED' if score < threshold else '✓ coherent'}")

    @staticmethod
    def narrative_arc(arc: str, themes: List[str]):
        print(f"\n  {C.NARR}Narrative arc: {arc.upper()}{C.RESET}  "
              f"{C.DIM}→ {' → '.join(themes)}{C.RESET}")

    @staticmethod
    def answer_box(result: "ThemeAlignedResult"):
        import textwrap
        print(f"\n{C.BOLD}{C.GREEN}{'═'*W}")
        print(f"  THEME-ALIGNED RAG ANSWER")
        print(f"{'═'*W}{C.RESET}")
        print(f"{C.BOLD}  Q: {result.query}{C.RESET}\n")
        for line in textwrap.wrap(result.answer, W - 4):
            print(f"  {line}")
        print(f"\n{C.DIM}{'─'*W}")
        mode_str = result.mode.value if isinstance(result.mode, RetrievalMode) else result.mode
        print(f"  Retrieval mode    : {mode_str}")
        print(f"  Themes used       : {result.themes_used}")
        print(f"  Chunks retrieved  : {len(result.retrieved_chunks)}")
        print(f"  Sources           : {result.unique_sources()}")
        if result.narrative_thread:
            print(f"  Narrative arc     : {result.narrative_thread.arc.value}")
        print(f"  Drift detected    : {result.drift_detected}"
              f"{'  (corrected)' if result.drift_corrected else ''}")
        print(f"  Retrieval rounds  : {result.retrieval_rounds}")
        print(f"  Elapsed           : {result.elapsed_sec:.2f}s")
        print(f"{'─'*W}{C.RESET}")


In [8]:
"""
════════════════════════════════
Two corpora:

  KNOWLEDGE_BASE (10 reference papers on topic modeling, RAG, NLG coherence)
  DEMO_DOCUMENTS (4 long, thematically rich documents spanning multiple themes)
    — Deliberately covers overlapping themes so the theme discovery and
      alignment machinery is meaningfully exercised.
    — Expected emergent themes (8 total across the corpus):
        T0: Attention & Transformer Architecture
        T1: Retrieval & Vector Search
        T2: Topic Modeling & Theme Discovery
        T3: Language Model Training & Scaling
        T4: Evaluation Metrics & Benchmarks
        T5: Memory & Knowledge Management
        T6: Coherence & Narrative Generation
        T7: Efficiency & Infrastructure

Primary PDF to download:
  BERTopic (Grootendorst, 2022)
  https://arxiv.org/pdf/2203.05794

📄 References:
  1.  BERTopic     https://arxiv.org/abs/2203.05794
  2.  LDA          https://dl.acm.org/doi/10.5555/944919.944937
  3.  TopicGPT     https://arxiv.org/abs/2311.01449
  4.  CSTM         https://dl.acm.org/doi/10.5555/1220175.1220243
  5.  Barzilay & Lapata 2008  https://doi.org/10.1162/coli.2008.34.1.1
  6.  Theme-RAG survey        https://arxiv.org/abs/2401.13192
  7.  CTRLsum      https://arxiv.org/abs/2012.04281
  8.  Topic-Aware Retrieval   https://arxiv.org/abs/2104.07597
  9.  RAG (Lewis et al.)      https://arxiv.org/abs/2005.11401
  10. REALM                   https://arxiv.org/abs/2002.08909
"""


'\n════════════════════════════════\nTwo corpora:\n\n  KNOWLEDGE_BASE (10 reference papers on topic modeling, RAG, NLG coherence)\n  DEMO_DOCUMENTS (4 long, thematically rich documents spanning multiple themes)\n    — Deliberately covers overlapping themes so the theme discovery and\n      alignment machinery is meaningfully exercised.\n    — Expected emergent themes (8 total across the corpus):\n        T0: Attention & Transformer Architecture\n        T1: Retrieval & Vector Search\n        T2: Topic Modeling & Theme Discovery\n        T3: Language Model Training & Scaling\n        T4: Evaluation Metrics & Benchmarks\n        T5: Memory & Knowledge Management\n        T6: Coherence & Narrative Generation\n        T7: Efficiency & Infrastructure\n\nPrimary PDF to download:\n  BERTopic (Grootendorst, 2022)\n  https://arxiv.org/pdf/2203.05794\n\n📄 References:\n  1.  BERTopic     https://arxiv.org/abs/2203.05794\n  2.  LDA          https://dl.acm.org/doi/10.5555/944919.944937\n  3.  Topic

In [9]:

from __future__ import annotations
from typing import Dict, List


# ══════════════════════════════════════════════════════════════════════
# KNOWLEDGE BASE — 10 reference papers
# ══════════════════════════════════════════════════════════════════════

KB_DOCS: List[Dict[str, str]] = [
    {
        "id": "bertopic",
        "source": "BERTopic: Neural Topic Modelling — Grootendorst (2022)",
        "url": "https://arxiv.org/abs/2203.05794",
        "content": """
BERTopic leverages BERT sentence embeddings and HDBSCAN clustering to discover
coherent topics in large document collections without requiring a fixed number of topics.

Pipeline:
  1. Embed documents with SBERT (sentence-transformers/all-MiniLM-L6-v2)
  2. Reduce dimensions with UMAP (from 384 → 5 dims for clustering)
  3. Cluster with HDBSCAN (density-based; does not require K a-priori)
  4. Represent topics via class-TF-IDF (c-TF-IDF): vocabulary weighted by
     how much a term differentiates a cluster vs the corpus background

c-TF-IDF formula: ctfidf(t,c) = tf(t,c) × log(1 + A/tf(t,C))
  where tf(t,c) = frequency in cluster c, tf(t,C) = frequency in corpus

BERTopic advantages over LDA:
  - Works on short texts (tweets, sentences) where LDA struggles
  - No need to specify K in advance (HDBSCAN finds natural clusters)
  - Uses contextual embeddings vs bag-of-words

Topic coherence (C_v score): 0.72 avg vs LDA 0.54 on 20NewsGroups.
Topic diversity: 0.89 vs LDA 0.71.
BERTopic supports dynamic topic modeling (evolution over time) and
hierarchical topic reduction.
"""
    },
    {
        "id": "lda",
        "source": "Latent Dirichlet Allocation — Blei et al. (2003)",
        "url": "https://dl.acm.org/doi/10.5555/944919.944937",
        "content": """
Latent Dirichlet Allocation (LDA) is a generative probabilistic model that
treats each document as a mixture of topics, and each topic as a distribution
over vocabulary words.

Generative process:
  For each document d: sample θ_d ~ Dirichlet(α)
  For each word position i in d:
    sample topic z_i ~ Multinomial(θ_d)
    sample word w_i  ~ Multinomial(φ_{z_i})

Inference: variational EM or Gibbs sampling to approximate the posterior
P(θ, z | w, α, β).

Key hyperparameters:
  α (document-topic prior): low α → documents focus on few topics
  β (topic-word prior):     low β → topics focus on few words
  K: number of topics (must be specified in advance)

Evaluation metrics:
  Perplexity: lower is better; measures held-out log-likelihood
  Topic coherence (PMI): higher means words in a topic co-occur more
  NPMI (normalized PMI): industry standard for topic coherence
  Typical NPMI values: LDA 0.10–0.15 on news corpora

LDA has been extended by: correlated topic models (CTM), author-topic
models, dynamic topic models (DTM), and neural topic models (ProdLDA, NTM).

In the Theme-Aligned RAG context: each document chunk gets a topic
distribution θ_chunk analogous to LDA's document-topic distribution.
"""
    },
    {
        "id": "topicgpt",
        "source": "TopicGPT: Topic Modeling with Large Language Models — Pham et al. (2023)",
        "url": "https://arxiv.org/abs/2311.01449",
        "content": """
TopicGPT uses an LLM to generate topic assignments and descriptions directly
from document text, rather than relying on statistical co-occurrence patterns.

Two-stage process:
  Stage 1 (Seed generation): LLM reads a random sample of documents and
    generates initial topic seeds with names and descriptions
  Stage 2 (Assignment + refinement): LLM assigns documents to seed topics
    and can create new topics when no existing topic fits well

Advantages over BERTopic/LDA:
  - Topics have human-readable, context-aware names from the start
  - Can handle rare topics not statistically prominent
  - Flexible topic granularity (coarse vs fine-grained topics)
  - No vocabulary or stopword lists needed

Limitations: cost (requires LLM calls per document), reproducibility
(LLM temperature > 0 → non-deterministic assignments), scalability.

On 20NewsGroups (20 topics): TopicGPT topics rated 4.1/5 by human judges
vs BERTopic 3.6/5 vs LDA 3.2/5 for coherence and interpretability.
TopicGPT enables Theme-Aligned RAG's LLM-based theme naming and description
generation step.
"""
    },
    {
        "id": "barzilay_lapata",
        "source": "Modeling Local Coherence — Barzilay & Lapata (2008)",
        "url": "https://doi.org/10.1162/coli.2008.34.1.1",
        "content": """
Barzilay and Lapata (2008) formalized local coherence in text as the pattern
of entity transitions across sentences. The entity grid model represents
a document as a matrix where rows are sentences and columns are entities.
Each cell contains the grammatical role (Subject, Object, None) of the entity.

Coherence is modeled by the distribution of entity transitions:
  e.g., SS (subject in two consecutive sentences) = high coherence
        NN (entity absent in two consecutive sentences) = low coherence

This work directly motivates Theme-Aligned RAG's narrative threading:
  - Themes play the role of entities: SS-equivalent = same theme across adjacent answer sections
  - Theme transitions must be smooth (topic sentences introduce theme shifts)
  - Incoherent retrievals flagged by abrupt theme changes in the retrieved set

Evaluation: entity grid coherence correlates with human judgments at r=0.58
(Spearman) on text ordering tasks. Models trained on coherence can reorder
shuffled sentences to near-original coherence 87% of the time.

Thematic coherence in RAG: if retrieved chunks form a theme sequence that
would be rated incoherent by the entity grid model (lots of NN transitions,
rare SS transitions), drift is detected and theme-constrained re-retrieval
is triggered.
"""
    },
    {
        "id": "theme_rag_survey",
        "source": "Theme-Guided Retrieval for Long-Form QA — Shen et al. (2024)",
        "url": "https://arxiv.org/abs/2401.13192",
        "content": """
Theme-guided retrieval augments standard dense retrieval with explicit thematic
structure, improving coherence in long-form question answering.

Key findings:
  1. Standard RAG retrieves thematically inconsistent chunks 41% of the time
     on long-form QA benchmarks (ELI5, ASQA, QMSum)
  2. Theme-guided retrieval reduces thematic inconsistency to 18%
  3. Human judges rate theme-guided answers as "more coherent" 67% of the time
     vs standard RAG answers
  4. ROUGE-L improves +3.2 points; BERTScore improves +2.1 points

Theme-guided pipeline:
  a. Extract themes from corpus using neural topic model
  b. Map each chunk to primary + secondary theme
  c. Align query to themes → select theme-coherent retrieval subset
  d. Generate answer using theme-structured prompt

Multi-theme queries (queries spanning multiple themes): the system retrieves
TOP_K_PER_THEME chunks from each relevant theme, then synthesizes across themes.
This produces answers that address all aspects of the query while maintaining
within-theme coherence.
"""
    },
    {
        "id": "ctrlsum",
        "source": "CTRLsum: Controlled Summarization with Keywords — He et al. (2022)",
        "url": "https://arxiv.org/abs/2012.04281",
        "content": """
CTRLsum enables controlled summarization by conditioning the generation on
a set of control keywords or aspects. This enables theme-directed summarization:
given a document and a theme keyword (e.g., "efficiency"), generate a summary
that focuses on the efficiency-related content.

Training: standard seq2seq summarization with randomly sampled keywords from
the oracle summary appended to the source document as a control prefix.
At inference: user-specified keywords guide generation to focus on desired aspects.

Results on CNN/DailyMail:
  Standard ROUGE-1: 43.7 (controlled) vs 43.9 (uncontrolled)
  Keyword-targeted ROUGE-1: 51.2 (controlled) vs 38.4 (uncontrolled)
  The controlled model achieves higher relevance on targeted aspects
  while maintaining comparable overall quality.

Application to Theme-Aligned RAG: the theme labels discovered for the corpus
serve as natural control keywords, guiding the LLM answer generation to stay
focused on the thematic aspects most relevant to the query.
"""
    },
    {
        "id": "topic_aware_retrieval",
        "source": "Topic-Aware Evidence Reasoning for Claim Verification — Zhao et al. (2021)",
        "url": "https://arxiv.org/abs/2104.07597",
        "content": """
Topic-aware evidence retrieval improves claim verification by ensuring retrieved
evidence is both semantically relevant and topically consistent with the claim.

Method:
  1. Extract topic from claim using LDA
  2. Retrieve candidate evidence passages via DPR
  3. Re-rank evidence by: α × semantic_score + (1−α) × topic_alignment
  4. Generate verdict using topic-aligned evidence set

Topic alignment score: cosine similarity between topic distribution vectors
of the claim and each evidence passage.

Results on FEVER:
  Topic-aware: 82.3% accuracy vs DPR-only 79.1%
  Oracle evidence: 91.2% (upper bound)

On HoVer (multi-hop): topic-aware 71.4% vs DPR-only 66.8%.
The topic alignment re-ranking particularly helps for claims that span multiple
topical domains — ensuring retrieved evidence does not stray into unrelated topics.

This directly implements the core Theme-Aligned RAG re-ranking formula:
  combined_score = α × semantic_score + (1−α) × theme_alignment_score
"""
    },
    {
        "id": "rag_lewis",
        "source": "RAG: Retrieval-Augmented Generation — Lewis et al. (2020)",
        "url": "https://arxiv.org/abs/2005.11401",
        "content": """
The original RAG paper combines a retrieval component (DPR) with a generative
model (BART) in an end-to-end trainable framework for open-domain QA.

Two variants:
  RAG-Sequence: retrieve once per full answer
  RAG-Token:    retrieve per generated token (fine-grained)

Training: jointly optimize the marginalized likelihood over retrieved documents.
DPR is updated during training; BART decoder receives all retrieved documents.

Results:
  NaturalQuestions: 44.5 EM vs T5-11B 36.6 (parametric-only)
  TriviaQA: 56.1 EM vs T5-11B 42.9
  WebQuestions: 45.5 EM vs T5-11B 37.4

Theme-Aligned RAG extends the RAG paradigm with:
  - Thematic pre-processing of the index
  - Query-theme alignment before retrieval
  - Theme-coherent candidate selection
  - Narrative threading across multiple themes
"""
    },
    {
        "id": "realm",
        "source": "REALM: Retrieval-Enhanced Language Model Pre-Training — Guu et al. (2020)",
        "url": "https://arxiv.org/abs/2002.08909",
        "content": """
REALM integrates retrieval into the language model pre-training process.
During pre-training, the model learns to retrieve relevant documents and use
them to improve masked language modeling predictions.

Architecture:
  Knowledge Retriever: BERT encoder embeds query → retrieves top-k docs
  Knowledge Augmented Encoder: BERT encoder processes query + retrieved doc
  Training: maximize log P(x_masked | x, z) marginalized over retrieved z

REALM is refreshed periodically by rebuilding the document index as the
retriever weights update. Index refresh every 500 training steps.

Open QA NaturalQuestions: REALM 40.4 EM vs GPT-3 29.9 vs T5-11B 36.6.
REALM demonstrates that retrieval can improve knowledge-intensive tasks even
when integrated at pre-training time, not just at inference.

Theme-Aligned RAG builds on REALM's insight that retrieval should be
semantically meaningful — and extends it to thematic meaningfulness:
the retrieved set should be not just relevant but thematically coherent.
"""
    },
    {
        "id": "cstm",
        "source": "Pachinko Allocation: DAG-Structured Mixture Models — Li & McCallum (2006)",
        "url": "https://dl.acm.org/doi/10.5555/1220175.1220243",
        "content": """
Pachinko Allocation (PAM) extends LDA with Directed Acyclic Graph (DAG) structure
over topics, allowing topics to have correlations and hierarchical relationships.
A super-topic (e.g., "Machine Learning") can have sub-topics
(e.g., "Supervised Learning", "Unsupervised Learning", "Reinforcement Learning").

The DAG structure enables:
  - Topic inheritance: sub-topic vocabulary includes super-topic vocabulary
  - Correlated topics: related topics share nodes in the DAG
  - Hierarchical browsing: navigate from coarse to fine-grained topics

Inference: Gibbs sampling over the DAG structure.
Perplexity on Reuters-21578: PAM 1480 vs LDA 1560 vs pLSA 1700.

In Theme-Aligned RAG: themes can have sub-themes (e.g., "Retrieval" has
sub-themes "Dense Retrieval", "Sparse Retrieval", "Hybrid Retrieval").
Queries about "vector search" would align to "Dense Retrieval" sub-theme,
propagating up to the "Retrieval" parent theme for cross-theme synthesis.
"""
    },
]


# ══════════════════════════════════════════════════════════════════════
# DEMO DOCUMENTS — 4 rich, multi-thematic documents
# ══════════════════════════════════════════════════════════════════════

DEMO_DOCUMENTS: List[Dict[str, str]] = [
    {
        "id": "transformer_evolution",
        "source": "The Transformer Revolution: Architecture, Training, and Scale",
        "url": "https://arxiv.org/abs/1706.03762",
        "content": """
The Transformer architecture (Vaswani et al., 2017) eliminated recurrence from
sequence modeling, replacing it with self-attention. The base Transformer has
65 million parameters: 6 encoder layers, 6 decoder layers, d_model=512, h=8
attention heads, d_ff=2048. Multi-head attention computes h parallel attention
functions with d_k=d_v=64.

Attention(Q,K,V) = softmax(QK^T / √d_k)V. The √d_k scaling prevents gradients
from vanishing due to large dot products. Position encodings use sinusoidal
functions: PE(pos,2i)=sin(pos/10000^{2i/d}), enabling the model to generalize
to sequence lengths unseen during training.

Training efficiency: the Transformer trains 3.5× faster than recurrent baselines
on 8 P100 GPUs. WMT 2014 EN→DE: 28.4 BLEU (Transformer-base), 28.4 (Transformer-large).
EN→FR: 41.8 BLEU, 0.5 BLEU higher than all prior single models.

BERT (Devlin et al., 2019) extended the Transformer encoder to bidirectional
pre-training using masked language modeling (MLM) and next sentence prediction (NSP).
BERT-large: 345M parameters, 24 layers, d_model=1024, h=16 heads.
BERT-base fine-tuned on SQuAD 1.1 achieves 88.5 F1, compared to human performance of 91.2.
BERT pre-training requires 4 days on 16 Cloud TPUs (v3-64 pod).

GPT-3 (Brown et al., 2020) scaled autoregressive language modeling to 175 billion
parameters. Training corpus: 300B tokens from Common Crawl (60%), WebText2 (22%),
Books1+2 (16%), Wikipedia (3%). Few-shot performance: TriviaQA 71.2% (5-shot),
NaturalQuestions 29.9% (1-shot). GPT-3 achieved 88.6% on the StoryCloze test.

Scaling laws (Kaplan et al., 2020): loss L ∝ (N/N_c)^{α_N} where N = parameters,
N_c ≈ 8.8×10^{13}, α_N ≈ 0.076. Compute-optimal training: C_opt = 6ND optimal tokens.
Chinchilla (Hoffmann et al., 2022) revised: compute-optimal = equal scaling of
parameters and tokens (D/N ≈ 20 tokens per parameter). Chinchilla 70B trained on
1.4T tokens outperforms Gopher 280B on 80% of tasks.
""",
    },
    {
        "id": "retrieval_systems",
        "source": "Modern Retrieval Systems: From Sparse to Dense to Hybrid",
        "url": "https://arxiv.org/abs/2004.04906",
        "content": """
BM25 (Robertson & Zaragoza, 2009) remains a strong sparse retrieval baseline.
BM25 score: Σ IDF(qi) × f(qi,D)(k1+1) / (f(qi,D)+k1(1-b+b|D|/avgdl)).
Typical parameters: k1=1.5, b=0.75. BM25 benefits from exact keyword matching
and is particularly strong for queries with rare, specific terms.

Dense retrieval (DPR, Karpukhin et al. 2020) uses dual BERT encoders. Questions
and passages embedded into the same 768-dimensional space. MIPS with FAISS IVF index.
NQ top-20: DPR 79.4% vs BM25 59.1%. DPR is weaker on short queries and queries
with exact keyword requirements. Index size: 21M passages × 768 dims × 4 bytes ≈ 61 GB.

ColBERT (Khattab & Zaharia, 2020) late interaction: MaxSim(q,d) = Σ max cosine similarity
between query token embeddings and document token embeddings. MS MARCO MRR@10: 39.7%
vs DPR 31.7% vs BM25 18.4%. ColBERT v2 adds residual compression: 2-bit quantization
reduces index size 6–8× with <1% quality loss.

Hybrid retrieval combines BM25 + dense scores: score_hybrid = α·score_dense + (1-α)·score_sparse.
BEIR benchmark (Thakur et al., 2021): hybrid retrieval outperforms either method alone
on 14/18 datasets. Optimal α varies by dataset (0.3–0.7).

Approximate Nearest Neighbor (ANN) algorithms:
  HNSW (Malkov & Yashunin, 2020): O(log n) search, M=16 neighbors per node.
    Recall@10 on SIFT-1M: 99.3% at 4ms query latency.
  IVF+PQ: 256 Voronoi cells, 64 product quantization subspaces.
    10× compression vs flat; <1% recall loss.
  ScaNN (Guo et al., 2020): anisotropic quantization, best recall-speed tradeoff.
    Outperforms HNSW by 2× throughput at equal recall on Google query workloads.

FAISS (Johnson et al., 2019): 1B-vector search at 300ms on 8 GPUs.
GPU-accelerated IVF+PQ: 100M vectors at <10ms, 1000 queries/second.
""",
    },
    {
        "id": "topic_modeling_eval",
        "source": "Topic Modeling Evaluation: Metrics, Methods, and Best Practices",
        "url": "https://arxiv.org/abs/2203.05794",
        "content": """
Evaluating topic models requires both intrinsic (corpus-based) and extrinsic
(task-based) metrics. The most widely used intrinsic metrics are perplexity,
topic coherence, and topic diversity.

Perplexity measures held-out likelihood: lower perplexity = better statistical
fit. However, perplexity often negatively correlates with human-judged topic
interpretability. A model with lower perplexity may produce less human-meaningful topics.
This is known as the perplexity-coherence tradeoff.

Topic coherence metrics:
  UCI (Newman et al., 2010): PMI-based, uses external Wikipedia corpus as reference.
    PMI(w_i, w_j) = log P(w_i,w_j) / (P(w_i)P(w_j))
  UMass (Mimno et al., 2011): uses document co-occurrence in the training corpus.
  C_v (Röder et al., 2015): uses a sliding window over the reference corpus.
    C_v is the most widely recommended; correlates best with human judgments (r=0.73).
  NPMI (Normalized PMI): NPMI(w_i,w_j) = PMI(w_i,w_j) / -log P(w_i,w_j).
    NPMI ranges from -1 (never co-occur) to +1 (always co-occur).
    Typical good topic: avg NPMI > 0.10.

Topic diversity: the fraction of unique words in the top-N words across all topics.
  diversity = |unique top words| / (K × N)
  High diversity (>0.70) indicates topics are distinct and cover different aspects.
  Low diversity (<0.40) suggests redundant topics.

Human evaluation protocols: topic word intrusion (identify a word that doesn't
belong to the topic), document intrusion (identify a document that doesn't fit
the topic's documents). Crowdsourcing via Amazon Mechanical Turk.

Best practices for topic number selection:
  - Run models with K ∈ {5, 10, 15, 20, 30, 50} and plot coherence vs K
  - Use elbow method on the coherence curve
  - Typical optimal K: 10–30 for news corpora, 5–15 for focused technical corpora
  - For Theme-Aligned RAG: K=8 is a reasonable default for mixed ML/NLP corpora
""",
    },
    {
        "id": "coherence_generation",
        "source": "Coherence in Text Generation: Theory, Models, and Applications",
        "url": "https://doi.org/10.1162/coli.2008.34.1.1",
        "content": """
Textual coherence refers to the property of a text where all parts contribute
meaningfully to a unified whole. Incoherent texts confuse readers because they
jump between unrelated topics without transitions.

Centering Theory (Grosz, Joshi & Weinstein, 1995) proposes that coherent discourse
focuses on a limited number of entities, each tracked across utterances via
backward-looking centers (CB) and forward-looking centers (CF).
Coherent transitions are ranked: CONTINUE > RETAIN > SHIFT > ROUGH-SHIFT.

Neural coherence models:
  Entity Grid (Barzilay & Lapata, 2008): encode entity transitions as feature vector,
    classify coherent vs incoherent. ACC on text ordering: 88.9% vs random 50%.
  Local Coherence Discriminator (Xu et al., 2019): sentence-pair coherence trained
    on contrastive examples. ACL 2019 best paper.
  BERT-based coherence (Jwalapuram et al., 2021): fine-tuned BERT achieves 92.1%
    on the sentence ordering task.

Coherence in RAG-generated answers:
  Problem: each retrieved chunk comes from a different document section. Simply
    concatenating chunks produces "mosaic" answers with jarring topic shifts.
  Solution: Theme-Aligned RAG uses narrative threading to impose coherence:
    (1) Identify dominant themes in the retrieved set
    (2) Assign chunks to thematic sections
    (3) Generate with a theme-ordered prompt and explicit transition sentences
    (4) Evaluate output coherence using entity grid or BERTScore

Transition sentences in multi-theme answers:
  Between themes T_i and T_j, a transition sentence bridges the two:
  "Having examined [T_i aspect], we now turn to [T_j aspect], which..."
  This mirrors the Centering Theory RETAIN pattern: the forward-looking center
  of the last T_i sentence becomes the backward-looking center of the first T_j sentence.

ROUGE and BERTScore as coherence proxies:
  ROUGE-L (Longest Common Subsequence): captures sequential coherence.
  BERTScore: uses contextual BERT embeddings; correlates with human coherence ratings
    at r=0.72, significantly higher than ROUGE-L (r=0.51).
""",
    },
]


# ══════════════════════════════════════════════════════════════════════
# METADATA
# ══════════════════════════════════════════════════════════════════════

PRIMARY_PDF = {
    "url":    "https://arxiv.org/pdf/2203.05794",
    "local":  "./bertopic_paper.pdf",
    "source": "BERTopic — Grootendorst (2022)",
    "url_ref":"https://arxiv.org/abs/2203.05794",
}

ALL_REFERENCES = [
    ("BERTopic (Grootendorst, 2022)",           "https://arxiv.org/abs/2203.05794"),
    ("LDA (Blei et al., 2003)",                  "https://dl.acm.org/doi/10.5555/944919.944937"),
    ("TopicGPT (Pham et al., 2023)",             "https://arxiv.org/abs/2311.01449"),
    ("Pachinko Allocation (Li & McCallum, 2006)","https://dl.acm.org/doi/10.5555/1220175.1220243"),
    ("Barzilay & Lapata (2008)",                 "https://doi.org/10.1162/coli.2008.34.1.1"),
    ("Theme-RAG Survey (Shen et al., 2024)",     "https://arxiv.org/abs/2401.13192"),
    ("CTRLsum (He et al., 2022)",                "https://arxiv.org/abs/2012.04281"),
    ("Topic-Aware Retrieval (Zhao et al., 2021)","https://arxiv.org/abs/2104.07597"),
    ("RAG (Lewis et al., 2020)",                 "https://arxiv.org/abs/2005.11401"),
    ("REALM (Guu et al., 2020)",                 "https://arxiv.org/abs/2002.08909"),
]


In [30]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import (
    EnsembleRetriever,
    ContextualCompressionRetriever,
)
from __future__ import annotations

import chromadb
from chromadb.config import Settings
import json
import math
import random
import time
import uuid
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate,SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import BaseOutputParser, StrOutputParser
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.runnables import RunnablePassthrough ,RunnableLambda
from sentence_transformers import CrossEncoder
from transformers import T5ForConditionalGeneration, T5Tokenizer

In [13]:
# ══════════════════════════════════════════════════════════════════════
# PROMPTS
# ══════════════════════════════════════════════════════════════════════

THEME_LABEL_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a topic modeling expert. Given a cluster of document excerpts, "
     "identify the unifying latent theme.\n"
     "Respond ONLY with valid JSON:\n"
     "{{\n"
     "  \"label\": \"Short Theme Name (3–6 words)\",\n"
     "  \"description\": \"1–2 sentence description of what this theme covers\",\n"
     "  \"top_words\": [\"keyword1\", \"keyword2\", ..., \"keyword12\"]\n"
     "}}"),
    ("human",
     "Cluster of {n_samples} document excerpts (sample):\n\n{samples}\n\n"
     "What is the unifying theme?"),
])

QUERY_THEME_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Analyze this query and determine which of the provided themes it most relates to.\n"
     "Assign a probability (0.0–1.0) to each theme. Probabilities must sum to 1.0.\n"
     "Consider: which themes contain information needed to answer this query?\n"
     "Respond ONLY with valid JSON:\n"
     "{{\"theme_probs\": {{\"T0\": 0.0, \"T1\": 0.0, ...}}, "
     "\"primary\": \"Tx\", \"reasoning\": \"1 sentence\"}}"),
    ("human",
     "Available themes:\n{theme_list}\n\nQuery: {query}\n\nTheme distribution:"),
])

In [14]:

# ══════════════════════════════════════════════════════════════════════
# EMBEDDING MANAGER
# ══════════════════════════════════════════════════════════════════════

class EmbeddingManager:
    _instance: Optional["EmbeddingManager"] = None

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self._model: Optional[HuggingFaceEmbeddings] = None

    @classmethod
    def get(cls, cfg: Config) -> "EmbeddingManager":
        if cls._instance is None:
            cls._instance = EmbeddingManager(cfg)
        return cls._instance

    def init(self) -> "EmbeddingManager":
        if self._model is None:
            Log.step("Loading HuggingFace Embeddings (local)")
            self._model = HuggingFaceEmbeddings(
                model_name=self.cfg.EMBEDDING_MODEL,
                model_kwargs={"device": self.cfg.EMBEDDING_DEVICE},
                encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
            )
            Log.ok("Embedding model ready")
        return self

    @property
    def model(self) -> HuggingFaceEmbeddings:
        if self._model is None:
            self.init()
        return self._model

    def embed_texts(self, texts: List[str]) -> List[List[float]]:
        return self.model.embed_documents(texts)

    def embed_query(self, text: str) -> List[float]:
        return self.model.embed_query(text)



In [15]:

# ══════════════════════════════════════════════════════════════════════
# VECTOR MATH HELPERS
# ══════════════════════════════════════════════════════════════════════

def _cosine_sim(a: List[float], b: List[float]) -> float:
    dot  = sum(x * y for x, y in zip(a, b))
    na   = math.sqrt(sum(x * x for x in a))
    nb   = math.sqrt(sum(x * x for x in b))
    return dot / (na * nb + 1e-9)


def _mean_vector(vecs: List[List[float]]) -> List[float]:
    if not vecs:
        return []
    dim = len(vecs[0])
    result = [0.0] * dim
    for v in vecs:
        for i, x in enumerate(v):
            result[i] += x
    n = len(vecs)
    # L2-normalize the centroid
    norm = math.sqrt(sum(x * x for x in result)) + 1e-9
    return [x / (n * norm) for x in result]


def _softmax(scores: List[float], temperature: float = 0.5) -> List[float]:
    """Temperature-scaled softmax. Lower τ → sharper distribution."""
    scaled = [s / temperature for s in scores]
    max_s  = max(scaled)
    exp_s  = [math.exp(s - max_s) for s in scaled]
    total  = sum(exp_s) + 1e-9
    return [e / total for e in exp_s]


def _entropy(dist: Dict[str, float]) -> float:
    """Shannon entropy of a probability distribution."""
    return -sum(p * math.log(p + 1e-9) for p in dist.values() if p > 0)


In [16]:


# ══════════════════════════════════════════════════════════════════════
# K-MEANS THEME CLUSTERER
# ══════════════════════════════════════════════════════════════════════

class KMeansThemeClusterer:
    """
    Lightweight K-means over embedding vectors to discover K theme centroids.
    Uses K-means++ initialization for better convergence.

    No external sklearn dependency — pure Python + math.
    For production: replace with BERTopic or sklearn.KMeans.
    """

    def __init__(self, k: int, max_iter: int = 30, random_seed: int = 42):
        self.k         = k
        self.max_iter  = max_iter
        self.rng       = random.Random(random_seed)
        self.centroids: List[List[float]] = []
        self.labels:    List[int]          = []

    def fit(self, embeddings: List[List[float]]) -> List[int]:
        """Run K-means and return cluster label for each embedding."""
        n = len(embeddings)
        if n < self.k:
            return list(range(n)) + [0] * (self.k - n)

        # K-means++ initialization
        self.centroids = [embeddings[self.rng.randint(0, n - 1)]]
        for _ in range(1, self.k):
            dists = [min(_cosine_dist(e, c) for c in self.centroids)
                     for e in embeddings]
            total = sum(dists)
            if total < 1e-9:
                self.centroids.append(embeddings[self.rng.randint(0, n - 1)])
                continue
            probs  = [d / total for d in dists]
            cum    = 0.0
            r      = self.rng.random()
            chosen = 0
            for i, p in enumerate(probs):
                cum += p
                if cum >= r:
                    chosen = i
                    break
            self.centroids.append(embeddings[chosen])

        # Iterate
        self.labels = [0] * n
        for iteration in range(self.max_iter):
            # Assignment
            new_labels = [
                min(range(self.k),
                    key=lambda j: _cosine_dist(e, self.centroids[j]))
                for e in embeddings
            ]
            if new_labels == self.labels and iteration > 0:
                break
            self.labels = new_labels

            # Update centroids
            for j in range(self.k):
                cluster_vecs = [embeddings[i] for i, l in enumerate(self.labels)
                                if l == j]
                if cluster_vecs:
                    self.centroids[j] = _mean_vector(cluster_vecs)

        return self.labels

    def soft_assign(
        self,
        embedding: List[float],
        temperature: float = 0.5,
    ) -> Dict[str, float]:
        """
        Compute soft theme distribution for a single embedding.
        P(theme_k | doc) = softmax(-cosine_dist(doc, centroid_k) / τ)
        """
        sims    = [_cosine_sim(embedding, c) for c in self.centroids]
        probs   = _softmax(sims, temperature)
        return {f"T{j}": p for j, p in enumerate(probs)}


def _cosine_dist(a: List[float], b: List[float]) -> float:
    return 1.0 - _cosine_sim(a, b)



In [17]:

# ══════════════════════════════════════════════════════════════════════
# THEME DISCOVERY ENGINE
# ══════════════════════════════════════════════════════════════════════

class ThemeDiscoveryEngine:
    """
    Discovers latent themes from the corpus and persists them to disk.
    On reload, themes and chunk distributions are loaded from JSON.

    Key operations:
      discover_themes()  — run clustering, label with LLM, compute coherence
      assign_chunk_dist()— compute ThemeDistribution for a single chunk
      assign_query_dist()— compute ThemeDistribution for a query
      load_or_discover() — load from disk if available, else discover
    """

    def __init__(
        self,
        cfg:     Config,
        em:      EmbeddingManager,
        llm:     Optional[ChatOllama] = None,
    ):
        self.cfg     = cfg
        self.em      = em
        self.llm     = llm
        self._parser = StrOutputParser()

        self.themes:    List[Theme]                 = []
        self.clusterer: Optional[KMeansThemeClusterer] = None
        # Cache of chunk_id → ThemeDistribution
        self._chunk_dists: Dict[str, ThemeDistribution] = {}

    # ── Persistence ────────────────────────────────────────────────────

    def save(self):
        Path(self.cfg.BASE_DIR).mkdir(parents=True, exist_ok=True)

        # Save themes (excluding centroid numpy — convert to list)
        themes_data = []
        for t in self.themes:
            d = {
                "theme_id":       t.theme_id,
                "label":          t.label,
                "description":    t.description,
                "top_words":      t.top_words,
                "coherence_score":t.coherence_score,
                "chunk_ids":      t.chunk_ids,
                "doc_sources":    t.doc_sources,
                "centroid":       t.centroid,
            }
            themes_data.append(d)
        with open(self.cfg.THEMES_PATH, "w") as f:
            json.dump(themes_data, f, indent=2)

        # Save chunk distributions
        dists_data = {
            cid: {
                "item_id":        td.item_id,
                "distribution":   td.distribution,
                "primary_theme":  td.primary_theme,
                "secondary_theme":td.secondary_theme,
                "entropy":        td.entropy,
            }
            for cid, td in self._chunk_dists.items()
        }
        with open(self.cfg.CHUNK_DIST_PATH, "w") as f:
            json.dump(dists_data, f, indent=2)

        Log.ok(f"Themes saved: {len(self.themes)} themes, "
               f"{len(self._chunk_dists)} chunk distributions")

    def load(self) -> bool:
        """Load themes + distributions from disk. Returns True if successful."""
        try:
            with open(self.cfg.THEMES_PATH) as f:
                themes_data = json.load(f)
            with open(self.cfg.CHUNK_DIST_PATH) as f:
                dists_data = json.load(f)

            self.themes = []
            for d in themes_data:
                t = Theme(
                    theme_id       = d["theme_id"],
                    label          = d["label"],
                    description    = d["description"],
                    top_words      = d["top_words"],
                    coherence_score= d["coherence_score"],
                    chunk_ids      = d["chunk_ids"],
                    doc_sources    = d["doc_sources"],
                    centroid       = d.get("centroid"),
                )
                self.themes.append(t)

            self._chunk_dists = {}
            for cid, d in dists_data.items():
                self._chunk_dists[cid] = ThemeDistribution(
                    item_id        = d["item_id"],
                    distribution   = d["distribution"],
                    primary_theme  = d["primary_theme"],
                    secondary_theme= d.get("secondary_theme"),
                    entropy        = d.get("entropy", 0.0),
                )

            # Rebuild clusterer from saved centroids
            if self.themes and self.themes[0].centroid:
                self.clusterer = KMeansThemeClusterer(k=len(self.themes))
                self.clusterer.centroids = [t.centroid for t in self.themes
                                             if t.centroid is not None]

            Log.ok(f"Themes loaded: {len(self.themes)} themes, "
                   f"{len(self._chunk_dists)} chunk distributions")
            return True
        except FileNotFoundError:
            return False
        except Exception as exc:
            Log.warn(f"Theme load failed: {exc}")
            return False

    # ── Theme Discovery ────────────────────────────────────────────────

    def discover_themes(
        self,
        chunk_texts: List[str],
        chunk_ids:   List[str],
        chunk_sources: List[str],
    ) -> List[Theme]:
        """
        Full theme discovery pipeline:
          1. Embed all chunks
          2. K-means clustering → K centroids
          3. LLM labels each cluster
          4. Compute soft distributions for all chunks
          5. Compute within-theme coherence
        """
        Log.section("THEME DISCOVERY", C.DISC)
        Log.step(f"Clustering {len(chunk_texts)} chunks into K={self.cfg.NUM_THEMES} themes")

        if not chunk_texts:
            Log.warn("No chunks to cluster")
            return []

        # ── Step 1: Embed ──────────────────────────────────────────────
        Log.info("Embedding all chunks…")
        embeddings = self.em.embed_texts(chunk_texts)
        Log.ok(f"Embedded {len(embeddings)} chunks ({len(embeddings[0])} dims)")

        # ── Step 2: K-means ────────────────────────────────────────────
        Log.info("Running K-means clustering…")
        self.clusterer = KMeansThemeClusterer(k=self.cfg.NUM_THEMES)
        labels         = self.clusterer.fit(embeddings)
        Log.ok(f"K-means done — {self.cfg.NUM_THEMES} clusters")

        # ── Step 3: LLM label each cluster ────────────────────────────
        Log.info("LLM labeling themes…")
        self.themes = []
        for k in range(self.cfg.NUM_THEMES):
            indices = [i for i, l in enumerate(labels) if l == k]
            if not indices:
                # Empty cluster — create placeholder
                self.themes.append(Theme(
                    theme_id=f"T{k}", label=f"Theme {k}",
                    description="Low-density cluster",
                    centroid=self.clusterer.centroids[k],
                ))
                continue

            cluster_texts   = [chunk_texts[i]   for i in indices]
            cluster_sources = [chunk_sources[i] for i in indices]
            cluster_embs    = [embeddings[i]     for i in indices]

            # Sample up to 5 chunks for LLM labeling
            samples  = "\n---\n".join(
                t[:300] for t in cluster_texts[:5]
            )
            label_data = self._label_cluster(
                samples, n_samples=len(cluster_texts)
            )

            # Centroid embedding
            centroid = _mean_vector(cluster_embs)

            theme = Theme(
                theme_id       = f"T{k}",
                label          = label_data.get("label", f"Theme {k}"),
                description    = label_data.get("description", ""),
                top_words      = label_data.get("top_words", []),
                centroid       = centroid,
                chunk_ids      = [chunk_ids[i] for i in indices],
                doc_sources    = list(set(cluster_sources)),
            )
            self.themes.append(theme)
            Log.theme_row(theme)

        # ── Step 4: Soft distributions for all chunks ─────────────────
        Log.step("Computing soft theme distributions for all chunks")
        for i, (chunk_id, emb) in enumerate(zip(chunk_ids, embeddings)):
            dist_dict = self.clusterer.soft_assign(emb)
            primary   = max(dist_dict, key=dist_dict.get)
            secondary = None
            sorted_t  = sorted(dist_dict.items(), key=lambda x: x[1], reverse=True)
            if (len(sorted_t) > 1 and
                    sorted_t[1][1] >= self.cfg.MULTI_THEME_THRESHOLD):
                secondary = sorted_t[1][0]

            self._chunk_dists[chunk_id] = ThemeDistribution(
                item_id        = chunk_id,
                distribution   = dist_dict,
                primary_theme  = primary,
                secondary_theme= secondary,
                entropy        = _entropy(dist_dict),
            )

        # ── Step 5: Within-theme coherence ────────────────────────────
        Log.step("Computing theme coherence scores")
        for theme in self.themes:
            theme_embs = [
                embeddings[chunk_ids.index(cid)]
                for cid in theme.chunk_ids
                if cid in chunk_ids
            ]
            if len(theme_embs) < 2:
                theme.coherence_score = 1.0
            else:
                sims = []
                for a in range(min(len(theme_embs), 20)):
                    for b in range(a + 1, min(len(theme_embs), 20)):
                        sims.append(_cosine_sim(theme_embs[a], theme_embs[b]))
                theme.coherence_score = sum(sims) / len(sims) if sims else 0.0

            flag = ("✓" if theme.coherence_score >= self.cfg.MIN_THEME_COHERENCE
                    else "⚠ low coherence")
            Log.info(f"  {theme.theme_id} '{theme.label}': "
                     f"coherence={theme.coherence_score:.3f} {flag}")

        Log.ok(f"Theme discovery complete: {len(self.themes)} themes, "
               f"{len(self._chunk_dists)} chunk distributions")
        return self.themes

    def _label_cluster(self, samples: str, n_samples: int) -> Dict[str, Any]:
        """Use LLM to label a cluster. Falls back to keyword extraction."""
        if self.llm is not None:
            try:
                chain  = THEME_LABEL_PROMPT | self.llm | self._parser
                raw    = chain.invoke({"samples": samples, "n_samples": n_samples})
                clean  = raw.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
                return json.loads(clean)
            except Exception as exc:
                Log.warn(f"LLM labeling failed ({exc}), using keyword fallback")

        # Keyword fallback: extract most common content words
        import re
        stopwords = {"the", "a", "an", "and", "or", "but", "in", "on", "at",
                     "to", "for", "of", "with", "by", "from", "is", "are",
                     "was", "were", "be", "been", "have", "has", "this", "that",
                     "which", "as", "it", "its", "their", "they", "we", "our"}
        words = re.findall(r'\b[a-z]{4,}\b', samples.lower())
        freq: Dict[str, int] = {}
        for w in words:
            if w not in stopwords:
                freq[w] = freq.get(w, 0) + 1
        top_words = sorted(freq, key=freq.get, reverse=True)[:12]
        label = " & ".join(w.capitalize() for w in top_words[:3]) if top_words else "Mixed"
        return {
            "label":       label,
            "description": f"Cluster covering content related to: {', '.join(top_words[:6])}",
            "top_words":   top_words,
        }

    # ── Query Theme Distribution ───────────────────────────────────────

    def assign_query_distribution(self, query: str) -> ThemeDistribution:
        """
        Compute theme distribution for a query.
        Primary method: embedding-based soft assignment (same as chunks).
        Secondary method: LLM-based (if configured) for validation.
        """
        if not self.clusterer or not self.clusterer.centroids:
            # No themes discovered yet
            k = self.cfg.NUM_THEMES
            uniform = {f"T{i}": 1.0/k for i in range(k)}
            return ThemeDistribution(
                item_id="query", distribution=uniform,
                primary_theme="T0", entropy=math.log(k),
            )

        emb      = self.em.embed_query(query)
        dist     = self.clusterer.soft_assign(emb)
        primary  = max(dist, key=dist.get)
        sorted_t = sorted(dist.items(), key=lambda x: x[1], reverse=True)
        secondary = (sorted_t[1][0]
                     if (len(sorted_t) > 1 and
                         sorted_t[1][1] >= self.cfg.MULTI_THEME_THRESHOLD)
                     else None)
        return ThemeDistribution(
            item_id        = "query",
            distribution   = dist,
            primary_theme  = primary,
            secondary_theme= secondary,
            entropy        = _entropy(dist),
        )

    def get_theme_by_id(self, theme_id: str) -> Optional[Theme]:
        for t in self.themes:
            if t.theme_id == theme_id:
                return t
        return None

    def get_chunk_distribution(self, chunk_id: str) -> Optional[ThemeDistribution]:
        return self._chunk_dists.get(chunk_id)


In [18]:


# ══════════════════════════════════════════════════════════════════════
# DOCUMENT INDEX WITH THEME METADATA
# ══════════════════════════════════════════════════════════════════════

class ThemeAlignedIndex:
    """
    ChromaDB collection storing chunks with theme distribution metadata.

    Metadata per chunk:
      doc_id, source, primary_theme, secondary_theme, theme_entropy,
      theme_T0..T7 (individual probabilities), page_num

    Retrieval returns ThemeAlignedChunk objects enriched with scores.
    """

    def __init__(self, cfg: Config, em: EmbeddingManager):
        self.cfg    = cfg
        self.em     = em
        self._store: Optional[Chroma] = None
        self._coll:  Optional[Any]    = None
        self._client: Optional[chromadb.PersistentClient] = None
        self._splitter = RecursiveCharacterTextSplitter(
            chunk_size=cfg.CHUNK_SIZE,
            chunk_overlap=cfg.CHUNK_OVERLAP,
            separators=["\n\n", "\n", ". ", " "],
        )

    def build(
        self,
        docs:      List[Dict[str, str]],
        discovery: ThemeDiscoveryEngine,
    ) -> Tuple[List[str], List[str], List[str]]:
        """
        Ingest documents, run theme discovery, build index.
        Returns (chunk_ids, chunk_texts, chunk_sources).
        """
        Path(self.cfg.CHROMA_DIR).mkdir(parents=True, exist_ok=True)

        # ── Chunk all documents ────────────────────────────────────────
        import hashlib
        all_chunks_text:    List[str] = []
        all_chunks_ids:     List[str] = []
        all_chunks_sources: List[str] = []
        all_chunks_meta:    List[Dict] = []

        for doc in docs:
            lc_docs = self._splitter.create_documents(
                [doc["content"].strip()],
                metadatas=[{"source": doc["source"], "doc_id": doc["id"],
                             "url": doc.get("url", "")}],
            )
            for pos, lc_doc in enumerate(lc_docs):
                text = lc_doc.page_content.strip()
                if len(text) < 60:
                    continue
                h  = hashlib.md5(text.encode()).hexdigest()[:10]
                cid = f"chk_{doc['id'][:6]}_{h}"
                if cid in all_chunks_ids:
                    continue
                all_chunks_ids.append(cid)
                all_chunks_text.append(text)
                all_chunks_sources.append(doc["source"])
                all_chunks_meta.append({
                    "doc_id":   doc["id"],
                    "source":   doc["source"],
                    "url":      doc.get("url", ""),
                    "position": pos,
                })

        Log.ok(f"Chunked {len(docs)} documents → {len(all_chunks_ids)} unique chunks")

        # ── Theme discovery ────────────────────────────────────────────
        discovery.discover_themes(
            chunk_texts   = all_chunks_text,
            chunk_ids     = all_chunks_ids,
            chunk_sources = all_chunks_sources,
        )
        discovery.save()

        # ── Build ChromaDB with theme metadata ─────────────────────────
        Log.step("Building theme-annotated ChromaDB index")
        self._client = chromadb.PersistentClient(
            path=self.cfg.CHROMA_DIR,
            settings=Settings(anonymized_telemetry=False),
        )
        try:
            self._client.delete_collection(self.cfg.COLLECTION_NAME)
        except Exception:
            pass

        lc_docs_out: List[Document] = []
        for cid, text, meta in zip(all_chunks_ids, all_chunks_text, all_chunks_meta):
            dist = discovery.get_chunk_distribution(cid)
            full_meta = dict(meta)
            full_meta["chunk_id"] = cid
            if dist:
                full_meta["primary_theme"]   = dist.primary_theme
                full_meta["secondary_theme"] = dist.secondary_theme or ""
                full_meta["theme_entropy"]   = round(dist.entropy, 4)
                # Store per-theme probabilities for filtering
                for tid, prob in dist.distribution.items():
                    full_meta[f"prob_{tid}"] = round(prob, 4)
            lc_docs_out.append(Document(page_content=text, metadata=full_meta))

        self._store = Chroma.from_documents(
            documents  = lc_docs_out,
            embedding  = self.em.model,
            client     = self._client,
            collection_name = self.cfg.COLLECTION_NAME,
            collection_metadata = {"hnsw:space": "cosine"},
        )
        n = self._store._collection.count()
        Log.ok(f"Index built: {n} theme-annotated chunks in ChromaDB")
        return all_chunks_ids, all_chunks_text, all_chunks_sources

    def load_existing(self):
        """Load an already-built index."""
        Path(self.cfg.CHROMA_DIR).mkdir(parents=True, exist_ok=True)
        self._client = chromadb.PersistentClient(
            path=self.cfg.CHROMA_DIR,
            settings=Settings(anonymized_telemetry=False),
        )
        self._store = Chroma(
            client          = self._client,
            collection_name = self.cfg.COLLECTION_NAME,
            embedding_function = self.em.model,
        )
        n = self._store._collection.count()
        Log.ok(f"Index loaded: {n} chunks")
        return n

    def semantic_retrieve(
        self,
        query: str,
        k: int,
        theme_filter: Optional[str] = None,
    ) -> List[Tuple[Document, float]]:
        """
        Semantic retrieval with optional primary_theme filter.
        Returns (document, score) pairs.
        """
        n = self._store._collection.count()
        if n == 0:
            return []
        actual_k = min(k, n)

        if theme_filter:
            # ChromaDB WHERE clause: only retrieve chunks whose primary theme matches
            try:
                self._coll = self._store._collection
                raw = self._coll.query(
                    query_texts=[query],
                    n_results=actual_k,
                    where={"primary_theme": {"$eq": theme_filter}},
                )
                if raw["ids"] and raw["ids"][0]:
                    results = []
                    for doc_text, meta, dist in zip(
                        raw["documents"][0],
                        raw["metadatas"][0],
                        raw["distances"][0],
                    ):
                        sim = max(0.0, 1.0 - float(dist))
                        results.append((Document(page_content=doc_text, metadata=meta), sim))
                    return results
            except Exception as exc:
                Log.warn(f"Theme filter query failed ({exc}), falling back to unfiltered")

        # Unfiltered MMR retrieval
        docs_mmr = self._store.max_marginal_relevance_search(
            query, k=actual_k,
            fetch_k=min(self.cfg.FETCH_K, n),
            lambda_mult=self.cfg.MMR_LAMBDA,
        )
        return [(doc, 0.85) for doc in docs_mmr]

    def count(self) -> int:
        return self._store._collection.count() if self._store else 0

    def try_pdf(self, discovery: ThemeDiscoveryEngine) -> bool:
        """Attempt to download and add primary PDF to index."""
        try:
            import urllib.request
            from langchain_community.document_loaders import PyPDFLoader

            if not Path(PRIMARY_PDF["local"]).exists():
                Log.info(f"Downloading {PRIMARY_PDF['url']}")
                urllib.request.urlretrieve(PRIMARY_PDF["url"], PRIMARY_PDF["local"])
                Log.ok(f"PDF saved → {PRIMARY_PDF['local']}")

            pages = PyPDFLoader(PRIMARY_PDF["local"]).load()
            Log.ok(f"PDF: {len(pages)} pages loaded")
            return True
        except Exception as exc:
            Log.warn(f"PDF unavailable ({exc})")
            return False


In [19]:
"""
═══════════════════════════════════════
Core retrieval and generation engine for Theme-Aligned RAG.

Implements five retrieval modes, all layered on top of the theme
distributions computed by ThemeDiscoveryEngine (theme_discovery.py):

  1. SEMANTIC_ONLY   — standard dense retrieval; baseline; no theme awareness
  2. THEME_ALIGNED   — retrieves chunks whose primary theme matches the query
  3. MULTI_THEME     — retrieves across all secondary themes, fuses by theme weight
  4. NARRATIVE       — builds a NarrativeThread: theme-ordered structure with transitions
  5. ADAPTIVE        — analyzes query, selects mode, detects drift, corrects if needed

Theme Drift Detection:
  After retrieval, compare query theme distribution to the mean theme
  distribution of retrieved chunks. If KL divergence > threshold, drift
  is detected → engine re-retrieves with explicit theme filter to correct.

Narrative Threading:
  Retrieved chunks are grouped by theme, ordered by NarrativeArc (e.g.
  PROBLEM_SOLUTION: themes ordered from problem-statement themes to
  solution themes). LLM generates transition sentences between sections.

Thematic Tension Synthesis:
  When chunks from contrasting themes co-occur, the engine generates
  a balanced synthesis that acknowledges both perspectives.

LLM Prompts:
  NARRATIVE_ARC_PROMPT    — choose arc and theme order for the answer
  TRANSITION_PROMPT       — generate inter-theme transition sentence
  ANSWER_PROMPT           — generate the final themed answer
  DRIFT_DIAGNOSIS_PROMPT  — explain and correct a detected theme drift
  TENSION_PROMPT          — synthesize two thematically opposed passages
  QUERY_EXPANSION_PROMPT  — expand query using discovered theme vocabulary

References:
  • Coherence & Discourse (Grosz & Sidner, 1986)
    https://doi.org/10.1111/j.1467-8640.1986.tb00188.x
  • RST (Mann & Thompson, 1988)
    https://doi.org/10.1177/0741088388005001003
  • BERTopic (Grootendorst, 2022)        https://arxiv.org/abs/2203.05794
  • LDA (Blei et al., 2003)             https://jmlr.org/papers/v3/blei03a.html
  • ThemeRAG (Lu et al., 2024)          https://arxiv.org/abs/2406.14834
  • HotpotQA (Yang et al., 2018)        https://arxiv.org/abs/1809.09600
"""

'\n═══════════════════════════════════════\nCore retrieval and generation engine for Theme-Aligned RAG.\n\nImplements five retrieval modes, all layered on top of the theme\ndistributions computed by ThemeDiscoveryEngine (theme_discovery.py):\n\n  1. SEMANTIC_ONLY   — standard dense retrieval; baseline; no theme awareness\n  2. THEME_ALIGNED   — retrieves chunks whose primary theme matches the query\n  3. MULTI_THEME     — retrieves across all secondary themes, fuses by theme weight\n  4. NARRATIVE       — builds a NarrativeThread: theme-ordered structure with transitions\n  5. ADAPTIVE        — analyzes query, selects mode, detects drift, corrects if needed\n\nTheme Drift Detection:\n  After retrieval, compare query theme distribution to the mean theme\n  distribution of retrieved chunks. If KL divergence > threshold, drift\n  is detected → engine re-retrieves with explicit theme filter to correct.\n\nNarrative Threading:\n  Retrieved chunks are grouped by theme, ordered by NarrativeAr

In [41]:
# ══════════════════════════════════════════════════════════════════════
# PROMPTS
# ══════════════════════════════════════════════════════════════════════

NARRATIVE_ARC_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Given a query and a list of themes active in the retrieved passages, "
     "choose the best narrative arc and theme ordering for the answer.\n\n"
     "Narrative arc options:\n"
     "  PROBLEM_SOLUTION: start with problem-framing themes, end with solution themes\n"
     "  COMPARE_CONTRAST: alternate between opposing themes\n"
     "  CHRONOLOGICAL: order themes by temporal progression\n"
     "  CAUSAL_CHAIN: order themes so each causes the next\n"
     "  GENERAL_SPECIFIC: broad themes first, specific ones last\n"
     "  FREE_FORM: best order for clarity\n\n"
     "Respond ONLY with valid JSON:\n"
     "{{\"arc\": \"PROBLEM_SOLUTION|COMPARE_CONTRAST|CHRONOLOGICAL|CAUSAL_CHAIN|"
     "GENERAL_SPECIFIC|FREE_FORM\", "
     "\"theme_order\": [\"theme_id_1\", \"theme_id_2\", ...], "
     "\"rationale\": \"brief reason\"}}"),
    ("human",
     "Query: {query}\n\n"
     "Active themes:\n{themes_list}"),
])

TRANSITION_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Write a single bridging sentence that transitions from one thematic section "
     "to the next in an essay-style answer. The sentence should:\n"
     "  • Briefly acknowledge the conclusion of the first section\n"
     "  • Introduce the perspective of the second section\n"
     "  • Be 1–2 sentences, fluent, and not formulaic\n"
     "Output ONLY the transition sentence, nothing else."),
    ("human",
     "Theme A: {theme_a}\n"
     "Theme B: {theme_b}\n"
     "Query context: {query}"),
])

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are answering a question using theme-aligned retrieved passages.\n"
     "The passages have been grouped by thematic thread and ordered "
     "according to the narrative arc.\n\n"
     "Rules:\n"
     "1. Follow the narrative thread order — address each thematic section in sequence.\n"
     "2. Use section headers drawn from theme labels (e.g., **Theme: Scale vs Efficiency**).\n"
     "3. Integrate transition sentences between sections.\n"
     "4. Cite sources inline as [Source Name].\n"
     "5. If themes are in tension, acknowledge both perspectives explicitly.\n"
     "6. Ground every claim in the provided passages.\n"
     "7. End with a synthesis that unifies the thematic threads.\n\n"
     "{theme_report}"),
    ("human",
     "Question: {query}\n\n"
     "Narrative thread:\n{narrative_context}"),
])

DRIFT_DIAGNOSIS_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Theme drift has been detected: the retrieved passages are thematically "
     "misaligned with the query.\n"
     "Diagnose the drift and suggest a corrected retrieval focus.\n"
     "Respond ONLY with JSON:\n"
     "{{\"diagnosis\": \"brief explanation\", "
     "\"correct_theme\": \"which theme label the query is really about\", "
     "\"corrected_query\": \"rewritten query emphasizing the correct theme\"}}"),
    ("human",
     "Original query: {query}\n"
     "Query primary theme: {query_theme}\n"
     "Retrieved chunks primary theme: {retrieved_theme}\n"
     "KL divergence: {kl_div:.3f}"),
])

TENSION_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Two retrieved passages express thematically opposing perspectives on the same topic. "
     "Write a balanced synthesis that:\n"
     "  • Accurately represents both positions\n"
     "  • Identifies the core point of disagreement\n"
     "  • Offers a nuanced reconciliation or explains the conditions under which "
     "each position holds\n"
     "Output ONLY the synthesis paragraph (3–5 sentences)."),
    ("human",
     "Topic: {topic}\n\n"
     "Position A [{source_a}]: {passage_a}\n\n"
     "Position B [{source_b}]: {passage_b}"),
])

QUERY_EXPANSION_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Expand this query with thematically relevant vocabulary to improve "
     "retrieval coverage. Use the active theme labels as a guide.\n"
     "Output ONLY the expanded query (1–2 sentences), nothing else."),
    ("human",
     "Original query: {query}\n"
     "Active themes: {theme_labels}"),
])

In [21]:

# ══════════════════════════════════════════════════════════════════════
# UTILITY — KL Divergence between theme distributions
# ══════════════════════════════════════════════════════════════════════

def kl_divergence(p: Dict[str, float], q: Dict[str, float]) -> float:
    """
    KL(P || Q) — measures how distribution P diverges from Q.
    Used for theme drift detection.
    Higher = more drift.
    """
    eps = 1e-10
    keys = set(p) | set(q)
    total = 0.0
    for k in keys:
        pk = p.get(k, eps)
        qk = q.get(k, eps)
        if pk > eps:
            total += pk * math.log(pk / (qk + eps))
    return max(0.0, total)


def mean_distribution(dists: List[ThemeDistribution]) -> Dict[str, float]:
    """Average multiple theme distributions into one."""
    if not dists:
        return {}
    all_keys: Set[str] = set()
    for d in dists:
        all_keys.update(d.distribution.keys())
    mean: Dict[str, float] = {}
    for k in all_keys:
        mean[k] = sum(d.distribution.get(k, 0.0) for d in dists) / len(dists)
    return mean


# ══════════════════════════════════════════════════════════════════════
# THEME ENGINE
# ══════════════════════════════════════════════════════════════════════

class ThemeEngine:
    """
    Theme-Aligned RAG engine.

    Orchestrates five retrieval modes, all grounded in the theme
    distributions computed by ThemeDiscoveryEngine over the corpus.
    """

    # KL divergence above this → theme drift detected
    DRIFT_KL_THRESHOLD = 0.40

    def __init__(self, cfg: Config):
        self.cfg      = cfg
        self._parser  = StrOutputParser()
        self.discovery: Optional[ThemeDiscoveryEngine]  = None
        self.index:     Optional[ThemeAlignedIndex]     = None
        self.em:        Optional[EmbeddingManager]      = None
        self.llm:       Optional[ChatOllama]       = None

    # ── Setup ──────────────────────────────────────────────────────────

    def setup(
        self,
        discovery: ThemeDiscoveryEngine,
        index:     ThemeAlignedIndex,
        em:        EmbeddingManager,
        llm:       Optional[ChatOllama] = None,
    ) -> "ThemeEngine":
        self.discovery = discovery
        self.index     = index
        self.em        = em
        self.llm       = llm
        Log.ok("ThemeEngine ready — all 5 retrieval modes available")
        return self

    def _invoke(self, prompt: ChatPromptTemplate, **kwargs) -> str:
        if self.llm is None:
            return "[LLM not configured — set OLLAMA_BASE_URL + OLLAMA_BASE_URL]"
        try:
            return (prompt | self.llm | self._parser).invoke(kwargs)
        except Exception as exc:
            Log.err(f"LLM: {exc}")
            return f"[LLM error: {exc}]"

    def _parse_json(self, raw: str, fallback: Any = None) -> Any:
        try:
            clean = raw.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
            return json.loads(clean)
        except Exception:
            return fallback or {}

    # ── Core: Analyze Query ────────────────────────────────────────────

    def analyze_query(self, query: str) -> QueryAnalysis:
        """
        Map the query into the discovered theme space.
        Returns a QueryAnalysis with primary theme, distribution, mode, and arc.
        """
        dist = self.discovery.assign_query_distribution(query)

        primary_t = self.discovery.get_theme_by_id(dist.primary_theme)
        secondary_ts = [
            self.discovery.get_theme_by_id(tid)
            for tid in (dist.secondary_theme,) if tid
        ]
        secondary_ts = [t for t in secondary_ts if t]

        # Determine status based on distribution entropy
        entropy = dist.entropy if hasattr(dist, 'entropy') else 0.5
        if entropy < 0.3:
            status = ThemeStatus.ALIGNED.value
        elif dist.secondary_theme:
            status = ThemeStatus.MULTI.value
        else:
            status = ThemeStatus.AMBIGUOUS.value

        # Choose retrieval mode
        if status == ThemeStatus.ALIGNED.value:
            mode = RetrievalMode.THEME_ALIGNED.value
        elif status == ThemeStatus.MULTI.value:
            mode = RetrievalMode.MULTI_THEME.value
        else:
            mode = RetrievalMode.ADAPTIVE.value

        # Choose narrative arc based on theme types
        arc = NarrativeArc.FREE_FORM.value
        if secondary_ts and len(secondary_ts) >= 2:
            arc = NarrativeArc.COMPARE_CONTRAST.value
        elif status == ThemeStatus.ALIGNED.value:
            arc = NarrativeArc.GENERAL_SPECIFIC.value

        # Expand query with theme vocabulary
        theme_labels = ", ".join(
            t.label for t in [primary_t] + secondary_ts if t
        )
        expanded = query
        if self.llm and theme_labels:
            expanded = self._invoke(
                QUERY_EXPANSION_PROMPT,
                query=query, theme_labels=theme_labels,
            )
            if expanded.startswith("[LLM"):
                expanded = query
            Log.info(f"Expanded query: {expanded[:80]}")

        return QueryAnalysis(
            query=query,
            theme_dist=dist,
            primary_theme=primary_t,
            secondary_themes=secondary_ts,
            status=status,
            retrieval_mode=mode,
            narrative_arc=arc,
            expanded_query=expanded,
        )

    # ── Core: Docs → ThemeAlignedChunks ───────────────────────────────

    def _docs_to_chunks(
        self, docs_scores: List[Tuple[Document, float]]
    ) -> List[ThemeAlignedChunk]:
        """Convert raw (Document, score) pairs to ThemeAlignedChunks."""
        chunks: List[ThemeAlignedChunk] = []
        for doc, sem_score in docs_scores:
            meta = doc.metadata
            cid  = meta.get("chunk_id", str(uuid.uuid4())[:10])
            dist = self.discovery.get_chunk_distribution(cid)
            chunks.append(ThemeAlignedChunk(
                chunk_id        = cid,
                content         = doc.page_content,
                doc_id          = meta.get("doc_id", ""),
                source          = meta.get("source", "?"),
                theme_dist      = dist,
                semantic_score  = sem_score,
                thematic_score  = 0.0,
                combined_score  = sem_score,
                page_num        = meta.get("position"),
                metadata        = meta,
            ))
        return chunks

    def _score_thematic(
        self,
        chunks: List[ThemeAlignedChunk],
        query_dist: ThemeDistribution,
    ) -> List[ThemeAlignedChunk]:
        """
        Compute thematic_score = dot product of query distribution and
        chunk distribution over the shared theme space.
        Combined = 0.5 * semantic + 0.5 * thematic.
        """
        for chunk in chunks:
            if chunk.theme_dist and query_dist:
                dot = sum(
                    query_dist.distribution.get(tid, 0.0) * prob
                    for tid, prob in chunk.theme_dist.distribution.items()
                )
                chunk.thematic_score = dot
                chunk.combined_score = 0.5 * chunk.semantic_score + 0.5 * dot
        # Sort by combined score descending
        chunks.sort(key=lambda c: c.combined_score, reverse=True)
        return chunks

    # ── Drift Detection ────────────────────────────────────────────────

    def _detect_drift(
        self,
        query_analysis: QueryAnalysis,
        chunks: List[ThemeAlignedChunk],
    ) -> Tuple[bool, float, str]:
        """
        Compare query theme distribution to the mean distribution of
        retrieved chunks. High KL divergence = theme drift.
        Returns (drift_detected, kl_value, corrected_query).
        """
        if not query_analysis.theme_dist or not chunks:
            return False, 0.0, query_analysis.query

        dists = [c.theme_dist for c in chunks if c.theme_dist]
        if not dists:
            return False, 0.0, query_analysis.query

        retrieved_mean = mean_distribution(dists)
        kl = kl_divergence(
            query_analysis.theme_dist.distribution, retrieved_mean
        )

        if kl < self.DRIFT_KL_THRESHOLD:
            return False, kl, query_analysis.query

        # Drift detected — diagnose and correct
        Log.warn(f"Theme drift detected — KL={kl:.3f} > threshold={self.DRIFT_KL_THRESHOLD}")

        # Find the top retrieved theme
        if retrieved_mean:
            top_retrieved = max(retrieved_mean, key=retrieved_mean.get)
            retrieved_theme_obj = self.discovery.get_theme_by_id(top_retrieved)
            retrieved_label = retrieved_theme_obj.label if retrieved_theme_obj else top_retrieved
        else:
            retrieved_label = "unknown"

        query_theme_label = (query_analysis.primary_theme.label
                              if query_analysis.primary_theme else "unknown")

        raw = self._invoke(
            DRIFT_DIAGNOSIS_PROMPT,
            query=query_analysis.query,
            query_theme=query_theme_label,
            retrieved_theme=retrieved_label,
            kl_div=kl,
        )
        data = self._parse_json(raw, {})
        corrected = data.get("corrected_query", query_analysis.query)
        diagnosis = data.get("diagnosis", "")
        Log.warn(f"Drift diagnosis: {diagnosis[:80]}")
        Log.info(f"Corrected query: {corrected[:80]}")

        return True, kl, corrected

    # ── Narrative Thread Builder ───────────────────────────────────────

    def _build_narrative_thread(
        self,
        query_analysis: QueryAnalysis,
        chunks: List[ThemeAlignedChunk],
    ) -> NarrativeThread:
        """
        Group chunks by primary theme, order themes by chosen arc,
        then generate transition sentences between theme sections.
        """
        # Group chunks by primary theme
        theme_groups: Dict[str, List[ThemeAlignedChunk]] = {}
        for chunk in chunks:
            tid = (chunk.theme_dist.primary_theme
                   if chunk.theme_dist else "unknown")
            theme_groups.setdefault(tid, []).append(chunk)

        active_theme_ids = list(theme_groups.keys())

        if not active_theme_ids:
            return NarrativeThread(
                arc=NarrativeArc(query_analysis.narrative_arc),
                theme_order=[], sections={}, transitions={},
            )

        # Ask LLM to determine arc + ordering
        themes_list = "\n".join(
            f"  {tid}: {self.discovery.get_theme_by_id(tid).label if self.discovery.get_theme_by_id(tid) else tid}"
            for tid in active_theme_ids
        )
        raw = self._invoke(
            NARRATIVE_ARC_PROMPT,
            query=query_analysis.query,
            themes_list=themes_list,
        )
        data   = self._parse_json(raw, {})
        arc_str = data.get("arc", query_analysis.narrative_arc)
        ordered = data.get("theme_order", active_theme_ids)

        # Fall back to whatever themes we have
        final_order = [tid for tid in ordered if tid in theme_groups]
        for tid in active_theme_ids:
            if tid not in final_order:
                final_order.append(tid)

        # Generate transition sentences
        transitions: Dict[str, str] = {}
        for i in range(len(final_order) - 1):
            tid_a = final_order[i]
            tid_b = final_order[i + 1]
            ta = self.discovery.get_theme_by_id(tid_a)
            tb = self.discovery.get_theme_by_id(tid_b)
            label_a = ta.label if ta else tid_a
            label_b = tb.label if tb else tid_b

            trans = self._invoke(
                TRANSITION_PROMPT,
                theme_a=label_a,
                theme_b=label_b,
                query=query_analysis.query,
            )
            transitions[f"{tid_a}→{tid_b}"] = trans
            Log.info(f"Transition [{label_a[:25]} → {label_b[:25]}]: {trans[:60]}…")

        try:
            arc_enum = NarrativeArc(arc_str.upper() if arc_str.islower() else arc_str)
        except ValueError:
            arc_enum = NarrativeArc.FREE_FORM

        return NarrativeThread(
            arc=arc_enum,
            theme_order=final_order,
            sections=theme_groups,
            transitions=transitions,
        )

    # ── Thematic Tension Detection ─────────────────────────────────────

    def _detect_tensions(
        self,
        chunks: List[ThemeAlignedChunk],
        query: str,
    ) -> List[Tuple[ThemeAlignedChunk, ThemeAlignedChunk, str]]:
        """
        Detect chunks from contrasting themes (high semantic similarity
        but different primary themes suggests thematic tension).
        Returns list of (chunk_a, chunk_b, synthesis).
        """
        tensions: List[Tuple[ThemeAlignedChunk, ThemeAlignedChunk, str]] = []
        checked: Set[frozenset] = set()

        for i, a in enumerate(chunks[:6]):
            for b in chunks[i+1:6]:
                pair = frozenset([a.chunk_id, b.chunk_id])
                if pair in checked:
                    continue
                checked.add(pair)

                # Tension = different primary themes but similar content
                if not (a.theme_dist and b.theme_dist):
                    continue
                if a.theme_dist.primary_theme == b.theme_dist.primary_theme:
                    continue
                # Check semantic similarity via embedding
                try:
                    emb_a = self.em.embed_texts([a.content[:200]])[0]
                    emb_b = self.em.embed_texts([b.content[:200]])[0]
                    sim = sum(x*y for x, y in zip(emb_a, emb_b))
                    if sim < 0.55:
                        continue
                except Exception:
                    continue

                # Generate synthesis
                synthesis = self._invoke(
                    TENSION_PROMPT,
                    topic=query,
                    source_a=a.source,
                    passage_a=a.content[:300],
                    source_b=b.source,
                    passage_b=b.content[:300],
                )
                if not synthesis.startswith("[LLM"):
                    tensions.append((a, b, synthesis))
                    Log.info(
                        f"Tension: [{a.source[:20]}] vs [{b.source[:20]}]"
                    )

        return tensions

    # ── Answer Generation ──────────────────────────────────────────────

    def _generate_answer(
        self,
        query_analysis: QueryAnalysis,
        chunks: List[ThemeAlignedChunk],
        thread: NarrativeThread,
        tensions: List[Tuple[ThemeAlignedChunk, ThemeAlignedChunk, str]],
    ) -> Tuple[str, str]:
        """
        Build the narrative context string and generate the final answer.
        Returns (answer, theme_report).
        """
        # Build narrative context with section headers + transitions
        narrative_parts: List[str] = []
        for i, tid in enumerate(thread.theme_order):
            t_obj = self.discovery.get_theme_by_id(tid)
            label = t_obj.label if t_obj else tid
            section_chunks = thread.sections.get(tid, [])

            narrative_parts.append(f"\n=== Theme: {label} ===")
            for c in section_chunks[:3]:
                narrative_parts.append(f"[{c.source}] {c.content[:400]}")

            # Add transition sentence
            if i < len(thread.theme_order) - 1:
                key = f"{tid}→{thread.theme_order[i+1]}"
                trans = thread.transitions.get(key, "")
                if trans:
                    narrative_parts.append(f"\n[Transition] {trans}")

        # Append tension syntheses
        if tensions:
            narrative_parts.append("\n=== Thematic Tensions ===")
            for a, b, synthesis in tensions:
                narrative_parts.append(
                    f"[{a.source} vs {b.source}] {synthesis}"
                )

        narrative_context = "\n".join(narrative_parts)

        # Build theme report
        theme_report = "Themes contributing to this answer:\n" + "\n".join(
            f"  • {self.discovery.get_theme_by_id(tid).label if self.discovery.get_theme_by_id(tid) else tid}"
            for tid in thread.theme_order
        )

        answer = self._invoke(
            ANSWER_PROMPT,
            query=query_analysis.query,
            narrative_context=narrative_context[:3000],
            theme_report=theme_report,
        )
        return answer, theme_report

    # ══════════════════════════════════════════════════════════════════
    # RETRIEVAL MODES
    # ══════════════════════════════════════════════════════════════════

    def semantic_only(self, query: str) -> ThemeAlignedResult:
        """
        Mode 1: SEMANTIC_ONLY — standard dense retrieval, no theme awareness.
        Baseline for comparison.
        """
        t0 = time.time()
        Log.section("SEMANTIC_ONLY (baseline)", C.CYAN)
        docs = self.index.semantic_retrieve(query, k=self.cfg.TOP_K_SEMANTIC)
        chunks = self._docs_to_chunks(docs)

        qa = QueryAnalysis(query=query)
        answer = self._invoke(
            ANSWER_PROMPT,
            query=query,
            narrative_context="\n\n".join(
                f"[{c.source}] {c.content[:400]}" for c in chunks[:6]
            ),
            theme_report="(no theme analysis in SEMANTIC_ONLY mode)",
        )
        return ThemeAlignedResult(
            query=query, mode=RetrievalMode.SEMANTIC_ONLY,
            query_analysis=qa, retrieved_chunks=chunks,
            answer=answer, elapsed_sec=round(time.time()-t0, 2),
        )

    def theme_aligned(self, query: str) -> ThemeAlignedResult:
        """
        Mode 2: THEME_ALIGNED — retrieve chunks whose primary theme
        matches the query's primary theme.
        Uses theme-filtered ChromaDB WHERE clause.
        """
        t0 = time.time()
        Log.section("THEME_ALIGNED retrieval", C.MESO)
        qa = self.analyze_query(query)

        if qa.primary_theme:
            Log.theme(qa.primary_theme, score=1.0)
            docs = self.index.semantic_retrieve(
                qa.expanded_query or query,
                k=self.cfg.TOP_K_THEME,
                theme_filter=qa.primary_theme.theme_id,
            )
        else:
            docs = self.index.semantic_retrieve(query, k=self.cfg.TOP_K_SEMANTIC)

        chunks = self._docs_to_chunks(docs)
        if qa.theme_dist:
            chunks = self._score_thematic(chunks, qa.theme_dist)

        # Drift check
        drift, kl, corrected = self._detect_drift(qa, chunks)
        if drift:
            Log.warn(f"Drift correcting with query: {corrected[:60]}")
            extra = self.index.semantic_retrieve(corrected, k=4)
            extra_chunks = self._docs_to_chunks(extra)
            chunks = (chunks + extra_chunks)[:self.cfg.TOP_K_THEME]

        answer, theme_report = self._generate_answer(
            qa, chunks,
            NarrativeThread(
                arc=NarrativeArc.FREE_FORM, theme_order=[], sections={}, transitions={}
            ),
            [],
        )
        return ThemeAlignedResult(
            query=query, mode=RetrievalMode.THEME_ALIGNED,
            query_analysis=qa, retrieved_chunks=chunks,
            answer=answer, theme_report=theme_report,
            drift_detected=drift, drift_corrected=drift,
            themes_used=[qa.primary_theme.theme_id] if qa.primary_theme else [],
            elapsed_sec=round(time.time()-t0, 2),
        )

    def multi_theme(self, query: str) -> ThemeAlignedResult:
        """
        Mode 3: MULTI_THEME — retrieve across all active themes weighted by
        the query's theme distribution, then re-rank by combined score.
        Enables rich answers that span multiple thematic perspectives.
        """
        t0 = time.time()
        Log.section("MULTI_THEME — spanning all active themes", C.MACRO)
        qa = self.analyze_query(query)

        all_chunks: List[ThemeAlignedChunk] = []
        themes_used: List[str] = []

        if qa.theme_dist:
            # Retrieve for each theme weighted above threshold
            for tid, prob in sorted(
                qa.theme_dist.distribution.items(), key=lambda x: -x[1]
            )[:4]:
                if prob < 0.1:
                    continue
                t_obj = self.discovery.get_theme_by_id(tid)
                Log.theme(t_obj, score=prob) if t_obj else None
                docs = self.index.semantic_retrieve(
                    qa.expanded_query or query,
                    k=max(2, int(self.cfg.TOP_K_THEME * prob * 2)),
                    theme_filter=tid,
                )
                new_chunks = self._docs_to_chunks(docs)
                all_chunks.extend(new_chunks)
                themes_used.append(tid)
        else:
            docs = self.index.semantic_retrieve(query, k=self.cfg.TOP_K_THEME)
            all_chunks = self._docs_to_chunks(docs)

        # Deduplicate by chunk_id
        seen: Set[str] = set()
        deduped: List[ThemeAlignedChunk] = []
        for c in all_chunks:
            if c.chunk_id not in seen:
                seen.add(c.chunk_id)
                deduped.append(c)

        if qa.theme_dist:
            deduped = self._score_thematic(deduped, qa.theme_dist)

        top_chunks = deduped[:self.cfg.TOP_K_THEME]

        # Build narrative thread
        thread = self._build_narrative_thread(qa, top_chunks)

        # Tension detection
        tensions = self._detect_tensions(top_chunks, query)

        answer, theme_report = self._generate_answer(qa, top_chunks, thread, tensions)

        return ThemeAlignedResult(
            query=query, mode=RetrievalMode.MULTI_THEME,
            query_analysis=qa, retrieved_chunks=top_chunks,
            narrative_thread=thread, answer=answer, theme_report=theme_report,
            themes_used=themes_used, elapsed_sec=round(time.time()-t0, 2),
        )

    def narrative(self, query: str) -> ThemeAlignedResult:
        """
        Mode 4: NARRATIVE — full thematic narrative threading.
        Retrieves broadly, then constructs a NarrativeThread with
        LLM-chosen arc, section grouping, and transition sentences.
        The answer reads as a structured essay, not a list of facts.
        """
        t0 = time.time()
        Log.section("NARRATIVE — theme-structured essay answer", C.THREAD)
        qa = self.analyze_query(query)

        # Broad retrieval: semantic + per-theme
        docs_sem = self.index.semantic_retrieve(
            qa.expanded_query or query,
            k=self.cfg.TOP_K_SEMANTIC,
        )
        all_chunks = self._docs_to_chunks(docs_sem)

        if qa.theme_dist:
            for tid, prob in sorted(
                qa.theme_dist.distribution.items(), key=lambda x: -x[1]
            )[:3]:
                if prob < 0.15:
                    continue
                extra = self.index.semantic_retrieve(
                    query, k=3, theme_filter=tid
                )
                all_chunks.extend(self._docs_to_chunks(extra))

        # Deduplicate
        seen: Set[str] = set()
        deduped: List[ThemeAlignedChunk] = []
        for c in all_chunks:
            if c.chunk_id not in seen:
                seen.add(c.chunk_id)
                deduped.append(c)

        if qa.theme_dist:
            deduped = self._score_thematic(deduped, qa.theme_dist)

        top_chunks = deduped[:self.cfg.TOP_K_THEME + 2]

        # Full narrative thread + transitions
        thread = self._build_narrative_thread(qa, top_chunks)
        tensions = self._detect_tensions(top_chunks, query)

        answer, theme_report = self._generate_answer(qa, top_chunks, thread, tensions)
        themes_used = list({
            c.theme_dist.primary_theme for c in top_chunks if c.theme_dist
        })

        return ThemeAlignedResult(
            query=query, mode=RetrievalMode.NARRATIVE,
            query_analysis=qa, retrieved_chunks=top_chunks,
            narrative_thread=thread, answer=answer, theme_report=theme_report,
            themes_used=themes_used, elapsed_sec=round(time.time()-t0, 2),
        )

    def adaptive(self, query: str) -> ThemeAlignedResult:
        """
        Mode 5: ADAPTIVE — analyzes the query, selects the best mode,
        runs it, detects drift, and corrects if needed.
        The default recommended mode.
        """
        t0 = time.time()
        Log.section("ADAPTIVE — auto-select mode + drift correction", C.MAG)
        qa = self.analyze_query(query)

        # Log query analysis
        if qa.primary_theme:
            Log.theme(qa.primary_theme, score=1.0)
        for t in qa.secondary_themes:
            Log.theme(t, score=0.5)
        Log.info(f"Status={qa.status}  Mode={qa.retrieval_mode}  Arc={qa.narrative_arc}")

        # Dispatch to appropriate sub-mode
        if qa.retrieval_mode == RetrievalMode.THEME_ALIGNED.value:
            sub_result = self.theme_aligned(query)
        elif qa.retrieval_mode == RetrievalMode.MULTI_THEME.value:
            sub_result = self.multi_theme(query)
        else:
            # Default: narrative for complex / ambiguous queries
            sub_result = self.narrative(query)

        # Patch mode to ADAPTIVE
        sub_result.mode = RetrievalMode.ADAPTIVE
        sub_result.elapsed_sec = round(time.time() - t0, 2)
        return sub_result

    # ── Main Dispatch ──────────────────────────────────────────────────

    def query(
        self,
        question: str,
        mode: RetrievalMode = RetrievalMode.ADAPTIVE,
    ) -> ThemeAlignedResult:
        Log.banner(
            f"THEME-ALIGNED RAG — {mode.value.upper()}",
            f"'{question[:65]}'",
        )
        dispatch = {
            RetrievalMode.SEMANTIC_ONLY:  self.semantic_only,
            RetrievalMode.THEME_ALIGNED:  self.theme_aligned,
            RetrievalMode.MULTI_THEME:    self.multi_theme,
            RetrievalMode.NARRATIVE:      self.narrative,
            RetrievalMode.ADAPTIVE:       self.adaptive,
        }
        fn = dispatch.get(mode, self.adaptive)
        result = fn(question)

        # Log retrieved chunks
        Log.section("RETRIEVED CHUNKS (by theme)", C.THREAD)
        for c in result.retrieved_chunks[:6]:
            tid = c.theme_dist.primary_theme if c.theme_dist else "?"
            t_obj = self.discovery.get_theme_by_id(tid) if tid != "?" else None
            theme_label = t_obj.label[:30] if t_obj else tid[:20]
            score_str = f"combined={c.combined_score:.3f}" if c.combined_score else ""
            print(f"  {C.DIM}[{c.source[:22]}]{C.RESET} "
                  f"{C.THREAD}{theme_label}{C.RESET} "
                  f"{C.DIM}{score_str}{C.RESET}")
            print(f"    {c.content[:100]}…")

        # Log themes used
        if result.themes_used:
            Log.section("THEMES ACTIVE IN ANSWER", C.MACRO)
            for tid in result.themes_used:
                t_obj = self.discovery.get_theme_by_id(tid)
                if t_obj:
                    Log.theme(t_obj)

        Log.answer_box(result)
        return result


In [22]:

W = 76


# ══════════════════════════════════════════════════════════════════════
# DEMO QUERIES
# ══════════════════════════════════════════════════════════════════════

DEMO_QUERIES = [
    # 1. SEMANTIC_ONLY — baseline, no theme awareness
    (
        RetrievalMode.SEMANTIC_ONLY,
        "What are the main techniques used to improve language model performance?",
        "Baseline: semantic similarity only — theme-blind",
    ),
    # 2. THEME_ALIGNED — single primary theme filter
    (
        RetrievalMode.THEME_ALIGNED,
        "How does scaling compute and data affect model capabilities?",
        "Single-theme: query maps strongly to 'scale' theme — precision retrieval",
    ),
    # 3. MULTI_THEME — spans several theme perspectives
    (
        RetrievalMode.MULTI_THEME,
        "What are the trade-offs between model size, training efficiency, and benchmark performance?",
        "Multi-theme: bridges 'efficiency', 'scale', and 'evaluation' themes",
    ),
    # 4. NARRATIVE — thematically structured essay
    (
        RetrievalMode.NARRATIVE,
        "How did the field's understanding of what makes a good language model evolve "
        "from BERT through GPT-3 to instruction-tuned models?",
        "Narrative: CHRONOLOGICAL arc across multiple thematic threads",
    ),
    # 5. ADAPTIVE — query analysis drives mode selection + drift correction
    (
        RetrievalMode.ADAPTIVE,
        "Why do models that score well on benchmarks sometimes fail at real-world tasks, "
        "and how is the community addressing this gap?",
        "Adaptive: ambiguous query — auto-selects mode, detects drift, synthesizes tensions",
    ),
]

In [36]:
# ══════════════════════════════════════════════════════════════════════
# SYSTEM BUILDER
# ══════════════════════════════════════════════════════════════════════

def build_system(cfg: Config, force_rebuild: bool = False) -> ThemeEngine:
    Log.banner(
        "THEME-ALIGNED RAG",
        "Theme Discovery · Thematic Retrieval · Narrative Threading · Drift Correction",
    )

    # ── Embeddings ─────────────────────────────────────────────────────
    em = EmbeddingManager.get(cfg)
    em.init()

    # ── LLM ────────────────────────────────────────────────────────────
    llm = None
    if cfg.is_configured():
        Log.step("Connecting to Azure OpenAI")
        llm = ChatOllama(
            base_url=getattr(cfg, 'OLLAMA_BASE_URL', 'http://localhost:11434'),
            model=getattr(cfg, 'OLLAMA_MODEL', 'qwen2.5-coder:7b'),
            temperature=getattr(cfg, 'LLM_TEMPERATURE', 0.0),
            num_predict=getattr(cfg, 'LLM_MAX_TOKENS', 512),
        )
        Log.ok(f"LLM ready — {cfg.OLLAMA_MODEL}")
    else:
        Log.warn("Azure OpenAI not configured — themes will use embedding-based clustering only")

    # ── Theme Discovery Engine ─────────────────────────────────────────
    discovery = ThemeDiscoveryEngine(cfg, em, llm)

    # ── Theme-Aligned Index ────────────────────────────────────────────
    index = ThemeAlignedIndex(cfg, em)

    # Determine if we need to build or can load
    Path(cfg.BASE_DIR).mkdir(parents=True, exist_ok=True)
    themes_exist = Path(cfg.THEMES_PATH).exists() if hasattr(cfg, 'THEMES_PATH') else False
    # Fallback: check chroma dir has content
    chroma_path = Path(cfg.CHROMA_DIR)
    index_exists = chroma_path.exists() and any(chroma_path.iterdir()) if chroma_path.exists() else False

    if not force_rebuild and index_exists:
        Log.step("Loading existing theme index…")
        try:
            n = index.load_existing()
            discovery.load()
            Log.ok(f"Loaded: {n} chunks · {len(discovery.themes)} themes")
        except Exception as exc:
            Log.warn(f"Load failed ({exc}) — rebuilding")
            force_rebuild = True

    if force_rebuild or not index_exists:
        Log.step("Building theme index from corpus (this runs once, ~30–120s)…")

        # Try primary PDF
        index.try_pdf(discovery)

        # Build full index with theme discovery using notebook-defined corpora
        all_docs = DEMO_DOCUMENTS + KB_DOCS
        index.build(all_docs, discovery)

    # Log discovered themes
    Log.section("DISCOVERED THEME HIERARCHY", C.MACRO)
    for t in discovery.themes:
        Log.theme(t)

    # Log references
    Log.step("Reference papers:")
    for i, (title, url) in enumerate(ALL_REFERENCES, 1):
        Log.info(f"  [{i:2d}] {title}")
        Log.info(f"        {url}")

    # ── REFRAG Engine ─────────────────────────────────────────────────
    engine = ThemeEngine(cfg)
    engine.setup(discovery, index, em, llm)
    return engine

In [24]:

# ══════════════════════════════════════════════════════════════════════
# RUN MODES
# ══════════════════════════════════════════════════════════════════════

def run_demo(engine: ThemeEngine, n: int = 0):
    queries = DEMO_QUERIES if n == 0 else [DEMO_QUERIES[n - 1]]
    print(f"\n{C.BOLD}{C.CYAN}Running {len(queries)} Theme-Aligned RAG demo "
          f"quer{'y' if len(queries)==1 else 'ies'}…{C.RESET}")

    for i, (mode, query, note) in enumerate(queries, 1 if n == 0 else n):
        print(f"\n{C.BOLD}{C.YELLOW}{'▓'*W}")
        print(f"  Demo {i}: {note}")
        print(f"{'▓'*W}{C.RESET}")
        try:
            engine.query(query, mode=mode)
        except Exception as exc:
            Log.err(f"Demo {i} failed: {exc}")
        time.sleep(0.3)


def print_themes(engine: ThemeEngine):
    Log.section("DISCOVERED THEME HIERARCHY", C.MACRO)
    if not engine.discovery or not engine.discovery.themes:
        Log.warn("No themes discovered yet — run with --rebuild first")
        return
    for t in engine.discovery.themes:
        doc_count = len(set(t.doc_sources)) if t.doc_sources else 0
        chunk_count = len(t.chunk_ids)
        print(f"  {C.BOLD}{t.label[:50]}{C.RESET}")
        print(f"    {C.DIM}id={t.theme_id}  coherence={t.coherence_score:.3f}  "
              f"docs={doc_count}  chunks={chunk_count}{C.RESET}")
        print(f"    {t.description[:100]}")
        print()


def run_interactive(engine: ThemeEngine):
    mode = RetrievalMode.ADAPTIVE

    print(f"\n{C.BOLD}{C.CYAN}")
    print("╔════════════════════════════════════════════════════════════╗")
    print("║     THEME-ALIGNED RAG — Interactive CLI                    ║")
    print("║  Commands:                                                  ║")
    print("║    'mode <n>'  — set retrieval mode:                     ║")
    print("║       semantic_only | theme_aligned | multi_theme |         ║")
    print("║       narrative | adaptive                                  ║")
    print("║    'themes'     — show discovered theme hierarchy           ║")
    print("║    'demo N'     — run demo query N (1–5)                    ║")
    print("║    'refs'       — print reference papers                    ║")
    print("║    'q' / 'quit' — exit                                      ║")
    print("╚════════════════════════════════════════════════════════════╝")
    print(C.RESET)

    while True:
        try:
            user = input(
                f"\n{C.BOLD}[{mode.value}] Query:{C.RESET} "
            ).strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break

        if not user:
            continue
        cmd = user.lower()

        if cmd in ("q", "quit", "exit"):
            break

        if cmd == "themes":
            print_themes(engine)
            continue

        if cmd == "refs":
            for i, (t, u) in enumerate(ALL_REFERENCES, 1):
                print(f"  [{i}] {t} — {u}")
            continue

        if cmd.startswith("mode "):
            name = user[5:].strip().lower()
            try:
                mode = RetrievalMode(name)
                Log.ok(f"Mode: {mode.value}")
            except ValueError:
                Log.warn(f"Unknown mode '{name}'. Valid: {[m.value for m in RetrievalMode]}")
            continue

        if cmd.startswith("demo"):
            parts = cmd.split()
            n = int(parts[1]) if len(parts) > 1 and parts[1].isdigit() else 0
            if n:
                run_demo(engine, n)
            else:
                for i, (m, q, note) in enumerate(DEMO_QUERIES, 1):
                    print(f"  {i}. [{m.value}] {q[:65]}…")
            continue

        try:
            engine.query(user, mode=mode)
        except Exception as exc:
            Log.err(f"Error: {exc}")



In [42]:
# Explicit query test run
# Compatibility patch in case older class definitions are already in memory
if not hasattr(C, "MACRO"):
    C.MACRO = C.THEME
if not hasattr(C, "MESO"):
    C.MESO = C.ALIGN
if not hasattr(C, "THREAD"):
    C.THREAD = C.NARR

def _theme_logger(theme, score=None):
    if theme is None:
        return
    if hasattr(theme, "theme_id") and hasattr(Log, "theme_row"):
        Log.theme_row(theme)
        if score is not None:
            print(f"{C.DIM}    score={score:.2f}{C.RESET}")
        return
    label = getattr(theme, "label", str(theme))
    if score is None:
        print(f"  {C.THEME}{label}{C.RESET}")
    else:
        print(f"  {C.THEME}{label}{C.RESET} {C.DIM}(score={score:.2f}){C.RESET}")

Log.theme = staticmethod(_theme_logger)

cfg = Config()
if not hasattr(cfg, "TOP_K_THEME"):
    cfg.TOP_K_THEME = cfg.TOP_K_FINAL
engine = build_system(cfg, force_rebuild=False)

test_query = "What are the trade-offs between model size, training efficiency, and benchmark performance?"
result = engine.query(test_query, mode=RetrievalMode.ADAPTIVE)

print("\n=== TEST QUERY ===")
print(test_query)
print("\n=== RESULT TYPE ===")
print(type(result).__name__)
if result is not None:
    print("\n=== RESULT PREVIEW ===")
    print(str(result)[:1200])


════════════════════════════════════════════════════════════════════════════
  THEME-ALIGNED RAG
  Theme Discovery · Thematic Retrieval · Narrative Threading · Drift Correction
════════════════════════════════════════════════════════════════════════════



▶ Connecting to Azure OpenAI
  ✓ LLM ready — qwen2.5-coder:7b

▶ Loading existing theme index…
  ✓ Index loaded: 41 chunks
  ✓ Themes loaded: 8 themes, 41 chunk distributions
  ✓ Loaded: 41 chunks · 8 themes

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  DISCOVERED THEME HIERARCHY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [T0] Thematic Retrieval             coh=0.63  chunks=  3  RAG · theme · coherence · retrieval · drift · alignment
  [T1] Neural Language Models         coh=0.46  chunks=  3  Transformer · BERT · GPT-3 · architecture · parameters · language modeling
  [T2] Text Coherence                 coh=0.42  chunks= 11  coherence · entity · theme · transitions · narrative · mosaic
  [T3] Topic Modeling Metrics and Evaluation coh=0.51  chunks=  5  topic modeling · metrics · perplexity · coherence · diversity · evaluation
  [T4] Information Retrieval Techniques coh=0.53  chunks=  4  BM25 · DPR · ColBERT · ANN · r